<a href="https://colab.research.google.com/github/somynt/Semester_4/blob/main/RAssignment280825.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade albumentations==1.3.1 --user

In [ ]:
import sys
sys.path.append('/usr/local/lib/python3.10/site-packages')

In [31]:
import os

# Define the mount point
MOUNT_POINT = '/content/drive'


In [ ]:
# Unzip the dataset file and store it in a folder called images.

!unzip "/content/drive/MyDrive/RM_Segmentation_Assignment_dataset.zip" -d "/content/drive/MyDrive/coco2017/"

Archive:  /content/drive/MyDrive/RM_Segmentation_Assignment_dataset.zip
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001751.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001380.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001643.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001583.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001685.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001482.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001494.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001551.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001654.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001594.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001411.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/000000001459.jpg  
  inflating: /content/drive/MyDrive/coco2017/test-30/00000000141

In [ ]:
# Define paths
# Use the correct path to your subset folder
DATASET_PATH = '/content/drive/MyDrive/RM_Segmentation_Assignment_dataset'
YOLO_OUTPUT_PATH = os.path.join(DATASET_PATH, 'yolo_data')

In [ ]:
!pip install --upgrade transformers accelerate
!pip install optuna
!pip install scikit-learn
!pip install seaborn
!pip install opencv-python
!pip install matplotlib seaborn
!pip install torch torchvision torchaudio
!pip install Pillow
!pip install tensorboard
!pip install ultralytics pycocotools tqdm
!pip install "optuna-integration[pytorch_lightning]"


In [33]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
from collections import Counter, defaultdict
from pycocotools.coco import COCO
from scipy.stats import chi2_contingency, pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx
from matplotlib.sankey import Sankey
import warnings
warnings.filterwarnings('ignore')
from scipy.spatial.distance import jensenshannon

# Enhanced plotting configuration for maximum readability from 1 meter distance
plt.rcParams.update({
    'font.size': 48,          # Extra large base font
    'axes.labelsize': 54,     # Very large axis labels
    'axes.titlesize': 60,     # Very large titles
    'xtick.labelsize': 44,    # Large tick labels
    'ytick.labelsize': 44,    # Large tick labels
    'legend.fontsize': 42,    # Large legend text
    'figure.titlesize': 66,   # Extra large figure titles
    'figure.autolayout': True,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'lines.linewidth': 4.0,   # Thicker lines for visibility
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 2.5,    # Thicker axis lines
    'grid.linewidth': 1.5,    # Visible grid lines
    'patch.linewidth': 2.0,   # Thicker patch borders
})
plt.style.use('seaborn-v0_8-whitegrid')

class EnhancedCOCOSegmentationEDA:
    """
    Advanced EDA for COCO Segmentation Dataset with comprehensive visualizations
    and statistical analysis to support data-driven insights.
    """
    def __init__(self, data_path, annotation_file, dataset_name="Dataset", target_class_names=None):
        self.data_path = data_path
        self.annotation_file = annotation_file
        self.dataset_name = dataset_name
        self.data = None
        self.coco = None

        # Load data first to populate self.data
        self.load_data()

        if target_class_names is None:
            self.target_class_names = [cat['name'] for cat in sorted(self.data['categories'], key=lambda x: x['id'])]
        else:
            self.target_class_names = target_class_names

        self.target_class_names_set = set(self.target_class_names)
        self.target_class_ids = [cat['id'] for cat in self.data['categories'] if cat['name'] in self.target_class_names_set]

        # Now, self.data exists, so these lines will work.
        self.cat_id_to_name = {cat['id']: cat['name'] for cat in self.data['categories']}
        self.cat_name_to_id = {cat['name']: cat['id'] for cat in self.data['categories']}
        self.image_id_to_info = {img['id']: img for img in self.data['images']}

        # Build other data structures.
        self._build_data_structures()

        print(f"Loaded {self.dataset_name}:")
        print(f"  - Images: {len(self.data['images'])}")
        print(f"  - Annotations: {len(self.data['annotations'])}")
        print(f"  - Target Categories: {len(self.target_class_names)}")

    def load_data(self):
        """Load and parse COCO annotations with error handling."""
        try:
            with open(self.annotation_file, 'r') as f:
                self.data = json.load(f)
            self.coco = COCO(self.annotation_file)
        except FileNotFoundError:
            print(f"Error: Annotation file not found at {self.annotation_file}")
            raise

    def _build_data_structures(self):
        """Build optimized data structures for analysis."""
        self.annotations_df = pd.DataFrame(self.data['annotations'])
        self.images_df = pd.DataFrame(self.data['images'])
        self.categories_df = pd.DataFrame(self.data['categories'])

        # Filter for target classes only
        target_anns = self.annotations_df[
            self.annotations_df['category_id'].isin(self.target_class_ids)
        ].copy()

        # Add class names
        target_anns['class_name'] = target_anns['category_id'].map(self.cat_id_to_name)
        self.target_annotations_df = target_anns

    def compute_class_weights(self, counts):
        """Compute class weights based on inverse frequency."""
        counts = np.array(counts, dtype=np.float32)
        counts[counts == 0] = 1.0
        inv = 1.0 / counts
        weights = inv / np.sum(inv) * len(counts)
        return weights

    def create_ultra_readable_pie_chart(self, sizes, labels, colors, title, output_dir, filename):
        """Create pie chart optimized for 1-meter viewing distance."""
        fig, ax = plt.subplots(figsize=(20, 16))  # Much larger figure

        wedges, _ = ax.pie(
            sizes,
            labels=None,
            colors=colors,
            startangle=90,
            wedgeprops=dict(width=0.6, edgecolor='white', linewidth=4),
            normalize=True
        )

        # Ultra-large annotations positioned for maximum visibility
        for i, (wedge, size, label) in enumerate(zip(wedges, sizes, labels)):
            angle = (wedge.theta2 + wedge.theta1) / 2
            x = np.cos(np.deg2rad(angle))
            y = np.sin(np.deg2rad(angle))
            ha = 'left' if x > 0 else 'right'
            va = 'center'

            pct = f"{size:.1f}%"
            label_text = f"{label}\n{pct}"

            ax.annotate(
                label_text,
                xy=(x * 0.7, y * 0.7),
                xytext=(1.6 * x, 1.6 * y),
                ha=ha, va=va,
                fontsize=48,  # Extra large font
                fontweight='bold',
                color='black',  # High contrast
                bbox=dict(boxstyle="round,pad=0.5", facecolor='white', alpha=0.9, edgecolor=colors[i], linewidth=3),
                arrowprops=dict(
                    arrowstyle="->",
                    color=colors[i],
                    lw=4,
                    connectionstyle=f"angle,angleA=0,angleB={angle}"
                )
            )

        plt.title(title, fontsize=66, fontweight='bold', pad=40)
        plt.axis('equal')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def analyze_dataset_overview(self, output_dir):
        """Comprehensive dataset overview with ultra-readable visualizations."""
        print(f"\n{'='*80}")
        print(f"COMPREHENSIVE DATASET OVERVIEW: {self.dataset_name}")
        print(f"{'='*80}")

        # 1. Dataset composition analysis
        all_image_ids = set(img['id'] for img in self.data['images'])
        target_image_ids = set(self.target_annotations_df['image_id'].unique())
        empty_images = len(all_image_ids - target_image_ids)
        populated_images = len(target_image_ids)

        print(f"Dataset Composition:")
        print(f"  - Total Images: {len(all_image_ids):,}")
        print(f"  - Images with Target Objects: {populated_images:,} ({populated_images/len(all_image_ids)*100:.1f}%)")
        print(f"  - Empty Images (no targets): {empty_images:,} ({empty_images/len(all_image_ids)*100:.1f}%)")

        # Ultra-readable pie chart for image composition
        sizes = [empty_images/len(all_image_ids)*100, populated_images/len(all_image_ids)*100]
        labels = ['Empty Images', 'Images with Targets']
        colors = ['#FF6B6B', '#4ECDC4']

        self.create_ultra_readable_pie_chart(
            sizes, labels, colors,
            f"{self.dataset_name}: Image Composition",
            output_dir, 'dataset_image_composition.png'
        )

        # 2. Object density histogram with ultra-large labels
        objects_per_image = self.target_annotations_df.groupby('image_id').size()

        fig, ax = plt.subplots(figsize=(20, 12))
        n, bins, patches = ax.hist(objects_per_image, bins=20, alpha=0.8, color='#45B7D1', edgecolor='black', linewidth=2)

        # Add value labels on top of each bar
        for i, (patch, count) in enumerate(zip(patches, n)):
            if count > 0:
                ax.text(patch.get_x() + patch.get_width()/2, patch.get_height() + max(n)*0.01,
                       f'{int(count)}', ha='center', va='bottom', fontsize=40, fontweight='bold')

        ax.set_xlabel('Number of Target Objects per Image', fontsize=54, fontweight='bold')
        ax.set_ylabel('Number of Images', fontsize=54, fontweight='bold')
        ax.set_title(f'{self.dataset_name}: Object Density Distribution', fontsize=60, fontweight='bold', pad=30)
        ax.grid(True, alpha=0.3, linewidth=2)
        ax.tick_params(labelsize=44)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'object_density_histogram.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def analyze_class_distribution_advanced(self, output_dir):
        """Advanced class distribution analysis with statistical insights."""
        print(f"\n{'='*80}")
        print(f"ADVANCED CLASS DISTRIBUTION ANALYSIS")
        print(f"{'='*80}")

        class_counts = self.target_annotations_df['class_name'].value_counts()
        class_image_counts = self.target_annotations_df.groupby('class_name')['image_id'].nunique()

        # Statistical analysis
        total_instances = class_counts.sum()
        entropy = -sum((count/total_instances) * np.log2(count/total_instances) for count in class_counts if count > 0)
        gini_coefficient = 1 - sum((count/total_instances)**2 for count in class_counts)

        print(f"Class Distribution Statistics:")
        print(f"  - Shannon Entropy: {entropy:.3f} (max: {np.log2(len(class_counts)):.3f})")
        print(f"  - Gini Coefficient: {gini_coefficient:.3f} (0=perfect equality, 1=perfect inequality)")
        print(f"  - Imbalance Ratio: {class_counts.max()/class_counts.min():.1f}:1")

        # Create comprehensive class analysis visualization
        fig, axes = plt.subplots(2, 2, figsize=(32, 24))
        fig.suptitle(f'{self.dataset_name}: Comprehensive Class Distribution Analysis',
                     fontsize=72, fontweight='bold', y=0.98)

        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#54A0FF', '#5F27CD']

        # 1. Instance counts with ultra-large labels
        bars1 = axes[0,0].bar(range(len(class_counts)), class_counts.values,
                             color=colors[:len(class_counts)], alpha=0.8, edgecolor='black', linewidth=2)
        axes[0,0].set_title('Instances per Class', fontsize=56, fontweight='bold', pad=20)
        axes[0,0].set_ylabel('Number of Instances', fontsize=48, fontweight='bold')
        axes[0,0].set_xticks(range(len(class_counts)))
        axes[0,0].set_xticklabels(class_counts.index, rotation=45, ha='right', fontsize=40)
        axes[0,0].tick_params(axis='y', labelsize=40)

        # Add value labels on bars
        for bar, count in zip(bars1, class_counts.values):
            axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(class_counts)*0.02,
                           f'{count:,}', ha='center', va='bottom', fontsize=36, fontweight='bold')

        # 2. Images per class
        bars2 = axes[0,1].bar(range(len(class_image_counts)), class_image_counts.values,
                             color=colors[:len(class_image_counts)], alpha=0.8, edgecolor='black', linewidth=2)
        axes[0,1].set_title('Images per Class', fontsize=56, fontweight='bold', pad=20)
        axes[0,1].set_ylabel('Number of Images', fontsize=48, fontweight='bold')
        axes[0,1].set_xticks(range(len(class_image_counts)))
        axes[0,1].set_xticklabels(class_image_counts.index, rotation=45, ha='right', fontsize=40)
        axes[0,1].tick_params(axis='y', labelsize=40)

        # Add value labels
        for bar, count in zip(bars2, class_image_counts.values):
            axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(class_image_counts)*0.02,
                           f'{count:,}', ha='center', va='bottom', fontsize=36, fontweight='bold')

        # 3. Class imbalance ratios with log scale
        max_count = class_counts.max()
        imbalance_ratios = [max_count / count for count in class_counts.values]

        bars3 = axes[1,0].bar(range(len(imbalance_ratios)), imbalance_ratios,
                             color=colors[:len(imbalance_ratios)], alpha=0.8, edgecolor='black', linewidth=2)
        axes[1,0].set_title('Class Imbalance Ratios', fontsize=56, fontweight='bold', pad=20)
        axes[1,0].set_ylabel('Imbalance Ratio (log scale)', fontsize=48, fontweight='bold')
        axes[1,0].set_yscale('log')
        axes[1,0].set_xticks(range(len(class_counts)))
        axes[1,0].set_xticklabels(class_counts.index, rotation=45, ha='right', fontsize=40)
        axes[1,0].tick_params(axis='y', labelsize=40)

        # Add ratio labels
        for bar, ratio in zip(bars3, imbalance_ratios):
            axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
                           f'{ratio:.1f}:1', ha='center', va='bottom', fontsize=32, fontweight='bold')

        # 4. Cumulative distribution
        sorted_counts = sorted(class_counts.values, reverse=True)
        cumulative_pct = np.cumsum(sorted_counts) / sum(sorted_counts) * 100

        axes[1,1].plot(range(1, len(cumulative_pct) + 1), cumulative_pct,
                       'o-', linewidth=4, markersize=12, color='#FF6B6B')
        axes[1,1].set_title('Cumulative Class Distribution', fontsize=56, fontweight='bold', pad=20)
        axes[1,1].set_xlabel('Class Rank', fontsize=48, fontweight='bold')
        axes[1,1].set_ylabel('Cumulative Percentage', fontsize=48, fontweight='bold')
        axes[1,1].grid(True, alpha=0.3, linewidth=2)
        axes[1,1].tick_params(labelsize=40)

        # Add percentage labels
        for i, pct in enumerate(cumulative_pct):
            axes[1,1].text(i+1, pct + 2, f'{pct:.1f}%', ha='center', va='bottom',
                           fontsize=32, fontweight='bold')

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'advanced_class_distribution.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def analyze_spatial_properties(self, output_dir):
        """Analyze spatial properties of annotations with advanced visualizations."""
        print(f"\n{'='*80}")
        print(f"SPATIAL PROPERTIES ANALYSIS")
        print(f"{'='*80}")

        # Extract spatial features
        spatial_data = []
        for _, ann in self.target_annotations_df.iterrows():
            bbox = ann['bbox']
            width, height = bbox[2], bbox[3]
            area = ann['area']
            aspect_ratio = width / height if height > 0 else 1

            spatial_data.append({
                'class_name': ann['class_name'],
                'area': area,
                'width': width,
                'height': height,
                'aspect_ratio': aspect_ratio,
                'bbox_area': width * height,
                'area_efficiency': area / (width * height) if (width * height) > 0 else 0
            })

        spatial_df = pd.DataFrame(spatial_data)

        # Create comprehensive spatial analysis
        fig, axes = plt.subplots(2, 3, figsize=(42, 28))
        fig.suptitle(f'{self.dataset_name}: Comprehensive Spatial Properties Analysis',
                     fontsize=72, fontweight='bold', y=0.98)

        classes = self.target_class_names
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3', '#54A0FF', '#5F27CD']

        # 1. Area distribution by class (violin plot)
        for i, cls in enumerate(classes):
            class_data = spatial_df[spatial_df['class_name'] == cls]['area']
            if len(class_data) > 0:
                parts = axes[0,0].violinplot([class_data], positions=[i], widths=0.6, showmeans=True)
                for pc in parts['bodies']:
                    pc.set_facecolor(colors[i % len(colors)])
                    pc.set_alpha(0.7)

        axes[0,0].set_title('Area Distribution by Class', fontsize=56, fontweight='bold', pad=20)
        axes[0,0].set_ylabel('Area (pixels²)', fontsize=48, fontweight='bold')
        axes[0,0].set_xticks(range(len(classes)))
        axes[0,0].set_xticklabels(classes, rotation=45, ha='right', fontsize=40)
        axes[0,0].tick_params(axis='y', labelsize=40)
        axes[0,0].set_yscale('log')
        axes[0,0].grid(True, alpha=0.3, linewidth=2)

        # 2. Aspect ratio distribution
        for i, cls in enumerate(classes):
            class_data = spatial_df[spatial_df['class_name'] == cls]['aspect_ratio']
            if len(class_data) > 0:
                axes[0,1].hist(class_data, bins=20, alpha=0.6, label=cls,
                              color=colors[i % len(colors)], edgecolor='black', linewidth=1)

        axes[0,1].set_title('Aspect Ratio Distribution', fontsize=56, fontweight='bold', pad=20)
        axes[0,1].set_xlabel('Width/Height Ratio', fontsize=48, fontweight='bold')
        axes[0,1].set_ylabel('Frequency', fontsize=48, fontweight='bold')
        axes[0,1].legend(fontsize=36, loc='upper right')
        axes[0,1].tick_params(labelsize=40)
        axes[0,1].grid(True, alpha=0.3, linewidth=2)

        # 3. Size vs efficiency scatter plot
        for i, cls in enumerate(classes):
            class_data = spatial_df[spatial_df['class_name'] == cls]
            if len(class_data) > 0:
                axes[0,2].scatter(class_data['bbox_area'], class_data['area_efficiency'],
                                 alpha=0.6, s=60, label=cls, color=colors[i % len(colors)])

        axes[0,2].set_title('Object Size vs Area Efficiency', fontsize=56, fontweight='bold', pad=20)
        axes[0,2].set_xlabel('Bounding Box Area', fontsize=48, fontweight='bold')
        axes[0,2].set_ylabel('Area Efficiency (mask/bbox)', fontsize=48, fontweight='bold')
        axes[0,2].set_xscale('log')
        axes[0,2].legend(fontsize=36)
        axes[0,2].tick_params(labelsize=40)
        axes[0,2].grid(True, alpha=0.3, linewidth=2)

        # 4. Average properties comparison
        avg_props = spatial_df.groupby('class_name').agg({
            'area': 'mean',
            'aspect_ratio': 'mean',
            'area_efficiency': 'mean'
        }).round(2)

        x = np.arange(len(classes))
        width = 0.25

        bars1 = axes[1,0].bar(x - width, [avg_props.loc[cls, 'area'] for cls in classes],
                             width, label='Avg Area', color='#4ECDC4', alpha=0.8, edgecolor='black')
        bars2 = axes[1,1].bar(x, [avg_props.loc[cls, 'aspect_ratio'] for cls in classes],
                             width, label='Avg Aspect Ratio', color='#FF6B6B', alpha=0.8, edgecolor='black')
        bars3 = axes[1,2].bar(x + width, [avg_props.loc[cls, 'area_efficiency'] for cls in classes],
                             width, label='Avg Efficiency', color='#45B7D1', alpha=0.8, edgecolor='black')

        # Configure subplots with ultra-large labels
        subplot_configs = [
            (axes[1,0], 'Average Area by Class', 'Area (pixels²)', bars1, [avg_props.loc[cls, 'area'] for cls in classes]),
            (axes[1,1], 'Average Aspect Ratio by Class', 'Width/Height Ratio', bars2, [avg_props.loc[cls, 'aspect_ratio'] for cls in classes]),
            (axes[1,2], 'Average Area Efficiency by Class', 'Efficiency Ratio', bars3, [avg_props.loc[cls, 'area_efficiency'] for cls in classes])
        ]

        for ax, title, ylabel, bars, values in subplot_configs:
            ax.set_title(title, fontsize=56, fontweight='bold', pad=20)
            ax.set_ylabel(ylabel, fontsize=48, fontweight='bold')
            ax.set_xticks(range(len(classes)))
            ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=40)
            ax.tick_params(axis='y', labelsize=40)
            ax.grid(True, alpha=0.3, linewidth=2)

            # Add value labels on bars
            for bar, value in zip(bars, values):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                       f'{value:.2f}', ha='center', va='bottom', fontsize=32, fontweight='bold')

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'spatial_properties_analysis.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

    def create_correlation_analysis(self, output_dir):
        """Create correlation matrix and advanced statistical analysis."""
        print(f"\n{'='*80}")
        print(f"CORRELATION AND STATISTICAL ANALYSIS")
        print(f"{'='*80}")

        # Prepare correlation data
        correlation_data = []
        for _, ann in self.target_annotations_df.iterrows():
            bbox = ann['bbox']
            width, height = bbox[2], bbox[3]
            area = ann['area']

            correlation_data.append({
                'area': area,
                'width': width,
                'height': height,
                'aspect_ratio': width/height if height > 0 else 1,
                'bbox_area': width * height,
                'area_efficiency': area/(width*height) if (width*height) > 0 else 0,
                'log_area': np.log10(area + 1),
                'perimeter_approx': 2 * (width + height)
            })

        corr_df = pd.DataFrame(correlation_data)
        correlation_matrix = corr_df.corr()

        # Create ultra-readable correlation heatmap
        fig, ax = plt.subplots(figsize=(20, 16))

        # Custom colormap for better visibility
        mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

        heatmap = sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
                              square=True, fmt='.3f', cbar_kws={"shrink": .8},
                              annot_kws={'fontsize': 36, 'fontweight': 'bold'},
                              linewidths=2, linecolor='white')

        ax.set_title(f'{self.dataset_name}: Spatial Properties Correlation Matrix',
                     fontsize=60, fontweight='bold', pad=30)

        # Ultra-large labels for 1-meter readability
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=42, fontweight='bold')
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=42, fontweight='bold')

        # Enhance colorbar
        cbar = heatmap.collections[0].colorbar
        cbar.ax.tick_params(labelsize=38)
        cbar.ax.set_ylabel('Correlation Coefficient', fontsize=48, fontweight='bold')

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'correlation_matrix.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

        # Statistical insights
        print("Key Correlations Found:")
        high_corr_pairs = []
        for i in range(len(correlation_matrix.columns)):
            for j in range(i+1, len(correlation_matrix.columns)):
                corr_val = correlation_matrix.iloc[i, j]
                if abs(corr_val) > 0.7:  # Strong correlation threshold
                    high_corr_pairs.append((
                        correlation_matrix.columns[i],
                        correlation_matrix.columns[j],
                        corr_val
                    ))

        for var1, var2, corr in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
            print(f"  - {var1} ↔ {var2}: {corr:.3f}")

    def compare_datasets_advanced(self, other_eda, output_dir, other_name="Validation"):
        """Advanced dataset comparison with statistical testing."""
        print(f"\n{'='*80}")
        print(f"ADVANCED DATASET COMPARISON: {self.dataset_name} vs {other_name}")
        print(f"{'='*80}")

        self_counts = self.target_annotations_df['class_name'].value_counts()
        other_counts = other_eda.target_annotations_df['class_name'].value_counts()

        # Ensure all classes are represented
        all_classes = set(self_counts.index) | set(other_counts.index)
        self_counts = self_counts.reindex(all_classes, fill_value=0)
        other_counts = other_counts.reindex(all_classes, fill_value=0)

        # Statistical testing
        chi2_stat, p_value = chi2_contingency([self_counts.values, other_counts.values])[:2]

        print(f"Statistical Comparison Results:")
        print(f"  - Chi-square statistic: {chi2_stat:.3f}")
        print(f"  - P-value: {p_value:.6f}")
        print(f"  - Significant difference: {'Yes' if p_value < 0.05 else 'No'} (α=0.05)")

        # Create comprehensive comparison visualization
        fig, axes = plt.subplots(2, 2, figsize=(32, 24))
        fig.suptitle(f'Advanced Dataset Comparison: {self.dataset_name} vs {other_name}',
                     fontsize=72, fontweight='bold', y=0.98)

        classes = sorted(all_classes)
        x = np.arange(len(classes))
        width = 0.35

        colors1 = ['#4ECDC4', '#FF6B6B', '#45B7D1', '#96CEB4', '#FECA57', '#FF9FF3']
        colors2 = ['#A8E6CF', '#FFB3B3', '#A8D8EA', '#C8E6C9', '#FFF3A0', '#FFB3E6']

        # 1. Absolute counts comparison
        bars1 = axes[0,0].bar(x - width/2, [self_counts[c] for c in classes], width,
                              label=self.dataset_name, alpha=0.8, color=colors1[:len(classes)],
                              edgecolor='black', linewidth=2)
        bars2 = axes[0,0].bar(x + width/2, [other_counts[c] for c in classes], width,
                              label=other_name, alpha=0.8, color=colors2[:len(classes)],
                              edgecolor='black', linewidth=2)

        axes[0,0].set_title('Instance Count Comparison', fontsize=56, fontweight='bold', pad=20)
        axes[0,0].set_xlabel('Class', fontsize=48, fontweight='bold')
        axes[0,0].set_ylabel('Number of Instances', fontsize=48, fontweight='bold')
        axes[0,0].set_xticks(x)
        axes[0,0].set_xticklabels(classes, rotation=45, ha='right', fontsize=40)
        axes[0,0].legend(fontsize=40, loc='upper right')
        axes[0,0].tick_params(axis='y', labelsize=40)
        axes[0,0].grid(True, alpha=0.3, linewidth=2)

        # Add value labels on bars
        for bars, counts in [(bars1, [self_counts[c] for c in classes]),
                             (bars2, [other_counts[c] for c in classes])]:
            for bar, count in zip(bars, counts):
                if count > 0:
                    axes[0,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() +
                                   max(max(self_counts), max(other_counts))*0.02,
                                   f'{count:,}', ha='center', va='bottom',
                                   fontsize=32, fontweight='bold')

        # 2. Percentage distribution comparison
        self_total = self_counts.sum()
        other_total = other_counts.sum()
        self_pct = [self_counts[c]/self_total*100 if self_total > 0 else 0 for c in classes]
        other_pct = [other_counts[c]/other_total*100 if other_total > 0 else 0 for c in classes]

        bars3 = axes[0,1].bar(x - width/2, self_pct, width, label=self.dataset_name,
                              alpha=0.8, color=colors1[:len(classes)], edgecolor='black', linewidth=2)
        bars4 = axes[0,1].bar(x + width/2, other_pct, width, label=other_name,
                              alpha=0.8, color=colors2[:len(classes)], edgecolor='black', linewidth=2)

        axes[0,1].set_title('Percentage Distribution Comparison', fontsize=56, fontweight='bold', pad=20)
        axes[0,1].set_xlabel('Class', fontsize=48, fontweight='bold')
        axes[0,1].set_ylabel('Percentage (%)', fontsize=48, fontweight='bold')
        axes[0,1].set_xticks(x)
        axes[0,1].set_xticklabels(classes, rotation=45, ha='right', fontsize=40)
        axes[0,1].legend(fontsize=40, loc='upper right')
        axes[0,1].tick_params(axis='y', labelsize=40)
        axes[0,1].grid(True, alpha=0.3, linewidth=2)

        # Add percentage labels
        for bars, pcts in [(bars3, self_pct), (bars4, other_pct)]:
            for bar, pct in zip(bars, pcts):
                if pct > 0:
                    axes[0,1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                                   f'{pct:.1f}%', ha='center', va='bottom',
                                   fontsize=32, fontweight='bold')

        # 3. Ratio comparison (Dataset1/Dataset2)
        ratios = []
        for c in classes:
            if other_counts[c] > 0:
                ratio = self_counts[c] / other_counts[c]
            elif self_counts[c] > 0:
                ratio = float('inf')
            else:
                ratio = 1.0
            ratios.append(ratio)

        # Cap infinite ratios for visualization
        display_ratios = [min(r, 10) if r != float('inf') else 10 for r in ratios]

        bars5 = axes[1,0].bar(classes, display_ratios, alpha=0.8,
                              color=colors1[:len(classes)], edgecolor='black', linewidth=2)
        axes[1,0].axhline(y=1, color='red', linestyle='--', linewidth=3, alpha=0.7)
        axes[1,0].set_title(f'Instance Ratio ({self.dataset_name}/{other_name})',
                            fontsize=56, fontweight='bold', pad=20)
        axes[1,0].set_xlabel('Class', fontsize=48, fontweight='bold')
        axes[1,0].set_ylabel('Ratio', fontsize=48, fontweight='bold')
        axes[1,0].tick_params(axis='x', rotation=45, labelsize=40)
        axes[1,0].tick_params(axis='y', labelsize=40)
        axes[1,0].grid(True, alpha=0.3, linewidth=2)

        # Add ratio labels
        for bar, ratio, orig_ratio in zip(bars5, display_ratios, ratios):
            label = f'{orig_ratio:.1f}' if orig_ratio != float('inf') else '∞'
            axes[1,0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(display_ratios)*0.02,
                           label, ha='center', va='bottom', fontsize=32, fontweight='bold')

        # 4. Distribution similarity analysis
        # Calculate KL divergence and other metrics

        self_dist = np.array(self_pct) / 100
        other_dist = np.array(other_pct) / 100

        # Add small epsilon to avoid log(0)
        epsilon = 1e-10
        self_dist_smooth = self_dist + epsilon
        other_dist_smooth = other_dist + epsilon

        js_distance = jensenshannon(self_dist_smooth, other_dist_smooth)

        # Visualization of distribution similarity
        axes[1,1].plot(range(len(classes)), self_pct, 'o-', linewidth=4, markersize=12,
                       label=self.dataset_name, color='#4ECDC4')
        axes[1,1].plot(range(len(classes)), other_pct, 's-', linewidth=4, markersize=12,
                       label=other_name, color='#FF6B6B')

        axes[1,1].set_title(f'Distribution Similarity (JS Distance: {js_distance:.3f})',
                            fontsize=56, fontweight='bold', pad=20)
        axes[1,1].set_xlabel('Class', fontsize=48, fontweight='bold')
        axes[1,1].set_ylabel('Percentage (%)', fontsize=48, fontweight='bold')
        axes[1,1].set_xticks(range(len(classes)))
        axes[1,1].set_xticklabels(classes, rotation=45, ha='right', fontsize=40)
        axes[1,1].legend(fontsize=40, loc='upper right')
        axes[1,1].tick_params(axis='y', labelsize=40)
        axes[1,1].grid(True, alpha=0.3, linewidth=2)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'advanced_dataset_comparison.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

        print(f"  - Jensen-Shannon Distance: {js_distance:.3f} (0=identical, 1=completely different)")
        return {
            'chi2_statistic': chi2_stat,
            'p_value': p_value,
            'js_distance': js_distance
        }

    def create_class_cooccurrence_analysis(self, output_dir):
        """Analyze and visualize class co-occurrence patterns."""
        print(f"\n{'='*80}")
        print(f"CLASS CO-OCCURRENCE ANALYSIS")
        print(f"{'='*80}")

        # Build co-occurrence matrix
        image_classes = self.target_annotations_df.groupby('image_id')['class_name'].apply(set).values
        classes = sorted(self.target_class_names)
        cooccurrence_matrix = np.zeros((len(classes), len(classes)))

        for class_set in image_classes:
            class_list = list(class_set)
            for i, class1 in enumerate(class_list):
                for j, class2 in enumerate(class_list):
                    if class1 in classes and class2 in classes:
                        idx1, idx2 = classes.index(class1), classes.index(class2)
                        cooccurrence_matrix[idx1, idx2] += 1

        # Normalize to get co-occurrence probabilities
        np.fill_diagonal(cooccurrence_matrix, 0)  # Remove self-occurrences
        row_sums = cooccurrence_matrix.sum(axis=1)
        normalized_matrix = np.divide(cooccurrence_matrix, row_sums[:, np.newaxis],
                                     out=np.zeros_like(cooccurrence_matrix),
                                     where=row_sums[:, np.newaxis]!=0)

        # Create ultra-readable co-occurrence heatmap
        fig, axes = plt.subplots(1, 2, figsize=(32, 14))
        fig.suptitle(f'{self.dataset_name}: Class Co-occurrence Analysis',
                     fontsize=72, fontweight='bold', y=0.98)

        # Raw counts heatmap
        sns.heatmap(cooccurrence_matrix, annot=True, fmt='.0f', cmap='Blues',
                   xticklabels=classes, yticklabels=classes, ax=axes[0],
                   annot_kws={'fontsize': 32, 'fontweight': 'bold'},
                   linewidths=2, linecolor='white', cbar_kws={"shrink": .8})

        axes[0].set_title('Co-occurrence Counts', fontsize=56, fontweight='bold', pad=20)
        axes[0].set_xlabel('Co-occurring Class', fontsize=48, fontweight='bold')
        axes[0].set_ylabel('Primary Class', fontsize=48, fontweight='bold')
        axes[0].tick_params(axis='both', labelsize=40, rotation=45)

        # Normalized probabilities heatmap
        sns.heatmap(normalized_matrix, annot=True, fmt='.3f', cmap='Reds',
                   xticklabels=classes, yticklabels=classes, ax=axes[1],
                   annot_kws={'fontsize': 32, 'fontweight': 'bold'},
                   linewidths=2, linecolor='white', cbar_kws={"shrink": .8})

        axes[1].set_title('Co-occurrence Probabilities', fontsize=56, fontweight='bold', pad=20)
        axes[1].set_xlabel('Co-occurring Class', fontsize=48, fontweight='bold')
        axes[1].set_ylabel('Primary Class', fontsize=48, fontweight='bold')
        axes[1].tick_params(axis='both', labelsize=40, rotation=45)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'class_cooccurrence_analysis.png'),
                    dpi=300, bbox_inches='tight', facecolor='white')
        plt.close(fig)

        # Print insights
        print("Key Co-occurrence Patterns:")
        for i, class1 in enumerate(classes):
            for j, class2 in enumerate(classes):
                if i != j and normalized_matrix[i, j] > 0.3:  # High co-occurrence threshold
                    print(f"  - {class1} → {class2}: {normalized_matrix[i, j]:.3f} probability")

    def generate_comprehensive_report(self, output_dir, other_eda=None, other_name="Validation"):
        """Generate a comprehensive EDA report with all analyses."""
        print(f"\n{'='*80}")
        print(f"GENERATING COMPREHENSIVE EDA REPORT")
        print(f"{'='*80}")

        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        # Run all analyses
        self.analyze_dataset_overview(output_dir)
        self.analyze_class_distribution_advanced(output_dir)
        self.analyze_spatial_properties(output_dir)
        self.create_correlation_analysis(output_dir)
        self.create_class_cooccurrence_analysis(output_dir)

        comparison_stats = None
        if other_eda:
            comparison_stats = self.compare_datasets_advanced(other_eda, output_dir, other_name)

        # Generate summary statistics
        summary_stats = self._generate_summary_statistics()

        # Save summary report
        report_path = os.path.join(output_dir, 'eda_summary_report.txt')
        with open(report_path, 'w') as f:
            f.write(f"COMPREHENSIVE EDA REPORT: {self.dataset_name}\n")
            f.write("="*80 + "\n\n")

            f.write("DATASET OVERVIEW:\n")
            f.write("-"*40 + "\n")
            for key, value in summary_stats.items():
                f.write(f"{key}: {value}\n")

            if other_eda and comparison_stats:
                f.write(f"\nCOMPARISON WITH {other_name}:\n")
                f.write("-"*40 + "\n")
                f.write(f"Chi-square p-value: {comparison_stats['p_value']:.6f}\n")
                f.write(f"Jensen-Shannon Distance: {comparison_stats['js_distance']:.3f}\n")

            f.write(f"\nGENERATED VISUALIZATIONS:\n")
            f.write("-"*40 + "\n")
            f.write("1. dataset_image_composition.png - Image composition analysis\n")
            f.write("2. object_density_histogram.png - Objects per image distribution\n")
            f.write("3. advanced_class_distribution.png - Comprehensive class analysis\n")
            f.write("4. spatial_properties_analysis.png - Spatial characteristics\n")
            f.write("5. correlation_matrix.png - Feature correlations\n")
            f.write("6. class_cooccurrence_analysis.png - Co-occurrence patterns\n")
            if other_eda:
                f.write("7. advanced_dataset_comparison.png - Dataset comparison\n")

        print(f"Comprehensive EDA report generated successfully!")
        print(f"All visualizations saved to: {output_dir}")
        print(f"Summary report saved to: {report_path}")

    def _generate_summary_statistics(self):
        """Generate comprehensive summary statistics."""
        total_images = len(set(img['id'] for img in self.data['images']))
        target_images = len(set(self.target_annotations_df['image_id']))
        total_annotations = len(self.target_annotations_df)

        class_counts = self.target_annotations_df['class_name'].value_counts()

        # Calculate spatial statistics
        areas = self.target_annotations_df['area'].values
        bbox_data = [ann['bbox'] for _, ann in self.target_annotations_df.iterrows()]
        widths = [bbox[2] for bbox in bbox_data]
        heights = [bbox[3] for bbox in bbox_data]
        aspect_ratios = [w/h if h > 0 else 1 for w, h in zip(widths, heights)]

        return {
            'Total Images': f"{total_images:,}",
            'Images with Target Objects': f"{target_images:,} ({target_images/total_images*100:.1f}%)",
            'Total Target Annotations': f"{total_annotations:,}",
            'Average Objects per Image': f"{total_annotations/target_images:.2f}",
            'Most Common Class': f"{class_counts.index[0]} ({class_counts.iloc[0]:,} instances)",
            'Class Imbalance Ratio': f"{class_counts.max()/class_counts.min():.1f}:1",
            'Average Object Area': f"{np.mean(areas):.0f} pixels²",
            'Median Object Area': f"{np.median(areas):.0f} pixels²",
            'Average Aspect Ratio': f"{np.mean(aspect_ratios):.2f}",
            'Area Standard Deviation': f"{np.std(areas):.0f} pixels²"
        }

def run_enhanced_target_class_eda():
    """Run comprehensive EDA analysis with ultra-readable visualizations."""
    output_dir = "enhanced_eda_plots"

    # Dataset paths - UPDATE THESE PATHS
    train_data_path = "/content/drive/MyDrive/coco2017/train-300/data"
    train_annotation_file = "/content/drive/MyDrive/coco2017/train-300/labels.json"
    val_annotation_file = "/content/drive/MyDrive/coco2017/validation-300/labels.json"

    target_class_names = ['car', 'person', 'dog', 'cake']

    print("="*80)
    print("ENHANCED COCO SEGMENTATION DATASET EDA")
    print("Ultra-Readable Visualizations for 1-Meter Viewing Distance")
    print("="*80)

    try:
        # Initialize training dataset analysis
        train_eda = EnhancedCOCOSegmentationEDA(
            train_data_path, train_annotation_file,
            "Training Set", target_class_names
        )

        # Try to load validation dataset
        val_eda = None
        try:
            val_eda = EnhancedCOCOSegmentationEDA(
                None, val_annotation_file,
                "Validation Set", target_class_names
            )
        except FileNotFoundError:
            print(f"\nNote: Validation file not found at {val_annotation_file}")
            print("Continuing with training set analysis only...")

        # Generate comprehensive report
        train_eda.generate_comprehensive_report(output_dir, val_eda, "Validation Set")

        print(f"\n{'='*80}")
        print("ENHANCED EDA ANALYSIS COMPLETE!")
        print(f"{'='*80}")
        print(f"All visualizations optimized for 1-meter viewing distance")
        print(f"Check the '{output_dir}' directory for all generated plots and reports")

    except FileNotFoundError as e:
        print(f"\nError: Could not load dataset files.")
        print(f"Please ensure the annotation files exist at the specified paths.")
        print(f"Training file: {train_annotation_file}")
        print(f"Validation file: {val_annotation_file}")
        raise

if __name__ == "__main__":
    run_enhanced_target_class_eda()

ENHANCED COCO SEGMENTATION DATASET EDA
Ultra-Readable Visualizations for 1-Meter Viewing Distance
loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Loaded Training Set:
  - Images: 300
  - Annotations: 3870
  - Target Categories: 4
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loaded Validation Set:
  - Images: 300
  - Annotations: 3774
  - Target Categories: 4

GENERATING COMPREHENSIVE EDA REPORT

COMPREHENSIVE DATASET OVERVIEW: Training Set
Dataset Composition:
  - Total Images: 300
  - Images with Target Objects: 300 (100.0%)
  - Empty Images (no targets): 0 (0.0%)

ADVANCED CLASS DISTRIBUTION ANALYSIS
Class Distribution Statistics:
  - Shannon Entropy: 1.078 (max: 2.000)
  - Gini Coefficient: 0.504 (0=perfect equality, 1=perfect inequality)
  - Imbalance Ratio: 103.8:1

SPATIAL PROPERTIES ANALYSIS

CORRELATION AND STATISTICAL ANALYSIS
Key Correlations Found:
  - area ↔ bbox_area: 1.000
  - height ↔ perimeter_appr

In [ ]:

# DATA CLEANING AND WRANGLING PIPELINE
# This script focuses on comprehensive data cleaning and preparation for future use

import os
import shutil
import json
import numpy as np
from collections import defaultdict
from tqdm.auto import tqdm
import pycocotools.mask as coco_mask
from concurrent.futures import ThreadPoolExecutor
import traceback
from typing import Dict, List, Tuple, Any
import datetime

# --- Configuration ---
# Set the path to your raw dataset
# Assumes a structure like:
# /content/drive/MyDrive/coco2017/
# ├── train-300/
# │   ├── data/
# │   └── labels.json
# └── validation-300/
#     ├── data/
#     └── labels.json
RAW_DATASET_PATH = '/content/drive/MyDrive/coco2017'

# Set the path where cleaned dataset will be saved
CLEANED_DATASET_PATH = '/content/drive/MyDrive/coco2017_cleaned'

# Target classes for your project
TARGET_CLASSES = ['cake', 'car', 'dog', 'person']

# Augmentation multipliers for future training
FUTURE_AUGMENTATION_MULTIPLIERS = {
    'person': 1,
    'car': 1,
    'dog': 96,
    'cake': 104
}

# Processing settings
MAX_WORKERS = min(8, (os.cpu_count() or 1))

print(f"🚀 DATA CLEANING AND WRANGLING PIPELINE")
print(f"📂 Raw dataset path: {RAW_DATASET_PATH}")
print(f"💾 Cleaned dataset will be saved to: {CLEANED_DATASET_PATH}")
print(f"🎯 Target classes: {TARGET_CLASSES}")
print(f"⚡ Using {MAX_WORKERS} workers for processing")

# --- Data Cleaning and Validation Functions ---
def clean_and_validate_dataset(coco_data: Dict, images_dir: str, target_classes: List[str], split_name: str) -> Dict:
    """
    Comprehensive data cleaning and validation pipeline.
    This is the CORE function for data preparation.
    """
    print(f"\n🧹 CLEANING {split_name.upper()} DATASET")
    print("="*60)

    # Statistics tracking
    stats = {
        'original_images': len(coco_data['images']),
        'original_annotations': len(coco_data['annotations']),
        'removed_invalid_polygons': 0,
        'removed_invalid_bboxes': 0,
        'removed_incomplete_data': 0,
        'removed_irrelevant_classes': 0,
        'removed_missing_images': 0,
        'removed_orphaned_annotations': 0,
        'removed_duplicates': 0,
        'corrected_data_types': 0,
        'clipped_coordinates': 0
    }

    print(f"📊 Original {split_name} stats:")
    print(f"  Images: {stats['original_images']}")
    print(f"  Annotations: {stats['original_annotations']}")
    print(f"  Categories: {len(coco_data['categories'])}")

    # Step 1: Filter relevant classes and create lookup maps
    target_class_ids = set()
    valid_categories = []
    class_mapping = {}

    for cat in coco_data['categories']:
        if cat['name'] in target_classes:
            target_class_ids.add(cat['id'])
            valid_categories.append(cat)
            class_mapping[cat['id']] = cat['name']

    print(f"🎯 Target classes found: {list(class_mapping.values())}")
    print(f"🎯 Target class IDs: {sorted(target_class_ids)}")

    # Step 2: Get list of existing image files
    existing_image_files = set()
    if os.path.exists(images_dir):
        all_files = os.listdir(images_dir)
        existing_image_files = {f.lower() for f in all_files
                                 if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'))}
        print(f"📁 Found {len(existing_image_files)} image files in directory")
    else:
        print(f"⚠️ Image directory not found: {images_dir}")

    # Step 3: Clean and validate images
    print("🖼️ Step 1/3: Validating images...")
    valid_images = []
    image_id_set = set()
    image_filename_set = set()

    for img in tqdm(coco_data['images'], desc="Processing images"):
        # Check for missing required fields
        if not all(field in img for field in ['id', 'file_name', 'width', 'height']):
            stats['removed_incomplete_data'] += 1
            continue

        # Check for duplicate image IDs
        if img['id'] in image_id_set:
            stats['removed_duplicates'] += 1
            continue
        image_id_set.add(img['id'])

        # Check for duplicate filenames
        if img['file_name'].lower() in image_filename_set:
            stats['removed_duplicates'] += 1
            continue
        image_filename_set.add(img['file_name'].lower())

        # Check if image file exists
        if img['file_name'].lower() not in existing_image_files:
            stats['removed_missing_images'] += 1
            continue

        # Validate image dimensions
        if not (isinstance(img.get('width'), (int, float)) and isinstance(img.get('height'), (int, float))):
            stats['removed_incomplete_data'] += 1
            continue

        if img['width'] <= 0 or img['height'] <= 0:
            stats['removed_invalid_bboxes'] += 1
            continue

        # Correct data types
        try:
            img['id'] = int(img['id'])
            img['width'] = int(img['width'])
            img['height'] = int(img['height'])
            stats['corrected_data_types'] += 1
        except (ValueError, TypeError):
            stats['removed_incomplete_data'] += 1
            continue

        valid_images.append(img)

    print(f"✅ Valid images: {len(valid_images)}/{stats['original_images']} "
          f"({100*len(valid_images)/max(1,stats['original_images']):.1f}%)")

    # Create set of valid image IDs for annotation filtering
    valid_image_ids = {img['id'] for img in valid_images}
    images_by_id = {img['id']: img for img in valid_images}

    # Step 4: Clean and validate annotations
    print("📝 Step 2/3: Validating annotations...")
    valid_annotations = []
    annotation_id_set = set()

    for ann in tqdm(coco_data['annotations'], desc="Processing annotations"):
        # Check for missing required fields
        required_fields = ['id', 'image_id', 'category_id', 'bbox', 'segmentation']
        if not all(field in ann for field in required_fields):
            stats['removed_incomplete_data'] += 1
            continue

        # Check for duplicate annotation IDs
        try:
            ann_id = int(ann['id'])
            if ann_id in annotation_id_set:
                stats['removed_duplicates'] += 1
                continue
            annotation_id_set.add(ann_id)
            ann['id'] = ann_id
        except (ValueError, TypeError):
            stats['removed_incomplete_data'] += 1
            continue

        # Filter irrelevant classes
        try:
            category_id = int(ann['category_id'])
            ann['category_id'] = category_id
        except (ValueError, TypeError):
            stats['removed_incomplete_data'] += 1
            continue

        if ann['category_id'] not in target_class_ids:
            stats['removed_irrelevant_classes'] += 1
            continue

        # Check if corresponding image exists
        try:
            image_id = int(ann['image_id'])
            ann['image_id'] = image_id
        except (ValueError, TypeError):
            stats['removed_incomplete_data'] += 1
            continue

        if ann['image_id'] not in valid_image_ids:
            stats['removed_orphaned_annotations'] += 1
            continue

        # Get image info for validation
        img_info = images_by_id[ann['image_id']]

        # Validate and clean bounding box
        bbox = ann['bbox']
        if not (isinstance(bbox, (list, tuple)) and len(bbox) == 4):
            stats['removed_invalid_bboxes'] += 1
            continue

        try:
            x, y, w, h = [float(coord) for coord in bbox]
        except (ValueError, TypeError):
            stats['removed_invalid_bboxes'] += 1
            continue

        # Check for invalid bbox dimensions
        if w <= 0 or h <= 0:
            stats['removed_invalid_bboxes'] += 1
            continue

        # Check and clip bbox to image bounds
        original_bbox = [x, y, w, h]
        x = max(0, min(x, img_info['width'] - 1))
        y = max(0, min(y, img_info['height'] - 1))
        w = max(1, min(w, img_info['width'] - x))
        h = max(1, min(h, img_info['height'] - y))

        if not np.allclose([x, y, w, h], original_bbox):
            stats['clipped_coordinates'] += 1

        ann['bbox'] = [x, y, w, h]

        # Validate segmentation
        segmentation = ann['segmentation']

        # Handle RLE format
        if isinstance(segmentation, dict):
            if 'counts' not in segmentation or 'size' not in segmentation:
                stats['removed_invalid_polygons'] += 1
                continue
            try:
                test_mask = coco_mask.decode(segmentation)
                if test_mask.shape != (img_info['height'], img_info['width']):
                    stats['removed_invalid_polygons'] += 1
                    continue
            except:
                stats['removed_invalid_polygons'] += 1
                continue

        # Handle polygon format
        elif isinstance(segmentation, list):
            valid_polygons = []
            for poly in segmentation:
                if not poly or len(poly) < 6:
                    continue
                try:
                    coords = [float(c) for c in poly]
                    if len(coords) % 2 != 0:
                        continue
                    points = np.array(coords).reshape(-1, 2)
                    if np.any(np.isnan(points)) or np.any(np.isinf(points)):
                        continue

                    original_points = points.copy()
                    points[:, 0] = np.clip(points[:, 0], 0, img_info['width'] - 1)
                    points[:, 1] = np.clip(points[:, 1], 0, img_info['height'] - 1)

                    if not np.array_equal(points, original_points):
                        stats['clipped_coordinates'] += 1

                    unique_points = []
                    tolerance = 1e-6
                    for i, point in enumerate(points):
                        if i == 0 or not np.allclose(point, points[i-1], atol=tolerance):
                            unique_points.append(point)

                    if len(unique_points) < 3:
                        continue
                    cleaned_poly = np.array(unique_points).flatten().tolist()
                    valid_polygons.append(cleaned_poly)
                except Exception:
                    continue
            if not valid_polygons:
                stats['removed_invalid_polygons'] += 1
                continue
            ann['segmentation'] = valid_polygons
        else:
            stats['removed_invalid_polygons'] += 1
            continue

        # Calculate area if missing or invalid
        if 'area' not in ann or ann.get('area', 0) <= 0:
            try:
                if isinstance(ann['segmentation'], dict):
                    mask = coco_mask.decode(ann['segmentation'])
                    ann['area'] = float(mask.sum())
                else:
                    ann['area'] = float(ann['bbox'][2] * ann['bbox'][3])
            except:
                ann['area'] = float(ann['bbox'][2] * ann['bbox'][3])
        else:
            ann['area'] = float(ann['area'])

        if 'iscrowd' not in ann:
            ann['iscrowd'] = 0
        else:
            ann['iscrowd'] = int(ann['iscrowd'])

        valid_annotations.append(ann)

    print(f"✅ Valid annotations: {len(valid_annotations)}/{stats['original_annotations']} "
          f"({100*len(valid_annotations)/max(1,stats['original_annotations']):.1f}%)")

    # Step 5: Remove images without annotations and final consistency check
    print("🔍 Step 3/3: Final consistency check...")
    images_with_annotations = {ann['image_id'] for ann in valid_annotations}
    final_images = [img for img in valid_images if img['id'] in images_with_annotations]

    if len(final_images) < len(valid_images):
        removed_count = len(valid_images) - len(final_images)
        print(f"🗑️ Removed {removed_count} images without valid annotations")
        stats['removed_missing_images'] += removed_count

    cleaned_data = {
        'images': final_images,
        'annotations': valid_annotations,
        'categories': valid_categories,
        'info': {
            'description': f'Cleaned COCO dataset - {split_name} split',
            'version': '1.0',
            'year': 2024,
            'contributor': 'Data Cleaning Pipeline',
            'date_created': datetime.datetime.now().isoformat(),
            'cleaning_stats': stats
        }
    }

    # Calculate and display final statistics
    print(f"\n📊 CLEANING SUMMARY FOR {split_name.upper()}:")
    print("="*50)
    print(f"🖼️  IMAGES:")
    print(f"    Original: {stats['original_images']:,}")
    print(f"    Final: {len(final_images):,}")
    print(f"    Retention: {100*len(final_images)/max(1,stats['original_images']):.1f}%")

    print(f"\n📝 ANNOTATIONS:")
    print(f"    Original: {stats['original_annotations']:,}")
    print(f"    Final: {len(valid_annotations):,}")
    print(f"    Retention: {100*len(valid_annotations)/max(1,stats['original_annotations']):.1f}%")

    print(f"\n🗑️  REMOVALS:")
    print(f"    Invalid polygons: {stats['removed_invalid_polygons']:,}")
    print(f"    Invalid bboxes: {stats['removed_invalid_bboxes']:,}")
    print(f"    Incomplete data: {stats['removed_incomplete_data']:,}")
    print(f"    Irrelevant classes: {stats['removed_irrelevant_classes']:,}")
    print(f"    Missing images: {stats['removed_missing_images']:,}")
    print(f"    Orphaned annotations: {stats['removed_orphaned_annotations']:,}")
    print(f"    Duplicates: {stats['removed_duplicates']:,}")

    print(f"\n🔧 CORRECTIONS:")
    print(f"    Data type fixes: {stats['corrected_data_types']:,}")
    print(f"    Clipped coordinates: {stats['clipped_coordinates']:,}")

    # Calculate class distribution
    class_counts = defaultdict(int)
    for ann in valid_annotations:
        class_name = class_mapping.get(ann['category_id'], 'unknown')
        class_counts[class_name] += 1

    print(f"\n📈 CLASS DISTRIBUTION:")
    total_instances = sum(class_counts.values())
    for class_name in sorted(class_counts.keys()):
        count = class_counts[class_name]
        percentage = 100 * count / max(1, total_instances)
        print(f"    {class_name}: {count:,} instances ({percentage:.1f}%)")

    return cleaned_data

def copy_images_batch(args: Tuple[str, str, List[str]]) -> int:
    """Copy a batch of images."""
    source_dir, target_dir, image_files = args
    success_count = 0
    for img_file in image_files:
        try:
            source_path = os.path.join(source_dir, img_file)
            target_path = os.path.join(target_dir, img_file)
            if os.path.exists(target_path):
                try:
                    if os.path.getsize(source_path) == os.path.getsize(target_path):
                        success_count += 1
                        continue
                except:
                    pass
            shutil.copy2(source_path, target_path)
            success_count += 1
        except Exception as e:
            print(f"⚠️ Failed to copy {img_file}: {e}")
    return success_count

def save_cleaned_dataset(cleaned_data: Dict, output_dir: str, split_name: str) -> Tuple[str, str, str]:
    """Save cleaned dataset to specified directory."""
    split_dir = os.path.join(output_dir, split_name)
    images_dir = os.path.join(split_dir, 'images')
    os.makedirs(images_dir, exist_ok=True)
    annotations_file = os.path.join(split_dir, 'annotations.json')
    class BytesEncoder(json.JSONEncoder):
        def default(self, obj):
            if isinstance(obj, bytes):
                return obj.decode('utf-8')
            elif isinstance(obj, np.integer):
                return int(obj)
            elif isinstance(obj, np.floating):
                return float(obj)
            return json.JSONEncoder.default(self, obj)
    with open(annotations_file, 'w') as f:
        json.dump(cleaned_data, f, cls=BytesEncoder, indent=2)
    print(f"💾 Saved cleaned annotations: {annotations_file}")
    return split_dir, images_dir, annotations_file

def create_dataset_summary(train_data: Dict, val_data: Dict, output_dir: str) -> None:
    """Create a comprehensive dataset summary."""
    summary = {
        'dataset_info': {
            'name': 'COCO2017 Cleaned Dataset',
            'target_classes': TARGET_CLASSES,
            'splits': ['train', 'validation'],
            'created': datetime.datetime.now().isoformat(),
            'source': RAW_DATASET_PATH,
            'cleaned_path': CLEANED_DATASET_PATH
        },
        'statistics': {},
        'class_distribution': {},
        'future_augmentation_plan': FUTURE_AUGMENTATION_MULTIPLIERS
    }
    for split_name, data in [('train', train_data), ('validation', val_data)]:
        summary['statistics'][split_name] = {
            'images': len(data['images']),
            'annotations': len(data['annotations']),
            'categories': len(data['categories'])
        }
        class_counts = defaultdict(int)
        cat_id_to_name = {cat['id']: cat['name'] for cat in data['categories']}
        for ann in data['annotations']:
            class_name = cat_id_to_name.get(ann['category_id'], 'unknown')
            class_counts[class_name] += 1
        summary['class_distribution'][split_name] = dict(class_counts)
        if 'info' in data and 'cleaning_stats' in data['info']:
            summary['statistics'][split_name]['cleaning_stats'] = data['info']['cleaning_stats']
    summary_file = os.path.join(output_dir, 'dataset_summary.json')
    with open(summary_file, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"📋 Dataset summary saved: {summary_file}")

    readme_content = f"""# COCO2017 Cleaned Dataset
## Overview
This dataset has been cleaned and prepared from the original COCO2017 dataset.
**Created:** {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
## Target Classes
{', '.join(TARGET_CLASSES)}
## Dataset Statistics
### Training Set
- Images: {summary['statistics']['train']['images']:,}
- Annotations: {summary['statistics']['train']['annotations']:,}
### Validation Set
- Images: {summary['statistics']['validation']['images']:,}
- Annotations: {summary['statistics']['validation']['annotations']:,}
## Class Distribution
### Training
"""
    for class_name, count in summary['class_distribution']['train'].items():
        readme_content += f"- {class_name}: {count:,} instances\n"
    readme_content += f"""
### Validation
"""
    for class_name, count in summary['class_distribution']['validation'].items():
        readme_content += f"- {class_name}: {count:,} instances\n"
    readme_content += f"""
## Future Augmentation Plan
For balanced training, the following augmentation multipliers are recommended:
"""
    for class_name, multiplier in FUTURE_AUGMENTATION_MULTIPLIERS.items():
        readme_content += f"- {class_name}: {multiplier}x multiplier\n"
    readme_content += f"""
## Directory Structure
{os.path.basename(CLEANED_DATASET_PATH)}/
├── train/
│   ├── images/       # Training images
│   └── annotations.json # Training annotations (COCO format)
├── validation/
│   ├── images/       # Validation images
│   └── annotations.json # Validation annotations (COCO format)
├── dataset_summary.json # Detailed statistics
└── README.md          # This file

## Data Cleaning Applied
- Removed corrupted/invalid annotations
- Filtered irrelevant classes
- Fixed data type mismatches
- Removed duplicate entries
- Validated bounding boxes and segmentations
- Clipped coordinates to image bounds
- Ensured image-annotation consistency
## Usage
This cleaned dataset is ready for:
1. Data augmentation
2. Model training
3. Computer vision experiments
4. Transfer learning
The data is in standard COCO format and can be used with popular frameworks like:
- Ultralytics YOLO
- Detectron2
- MMDetection
- PyTorch Vision
"""
    readme_file = os.path.join(output_dir, 'README.md')
    with open(readme_file, 'w') as f:
        f.write(readme_content)
    print(f"📖 README created: {readme_file}")

def main():
    """Main data cleaning and wrangling pipeline."""
    print("🚀 STARTING DATA CLEANING AND WRANGLING PIPELINE")
    print("="*60)
    print(f"📅 Started at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    print("\n🔍 VERIFYING SOURCE DATASET STRUCTURE")
    print("-" * 40)
    train_images_path = os.path.join(RAW_DATASET_PATH, 'train-300', 'data')
    train_labels_path = os.path.join(RAW_DATASET_PATH, 'train-300', 'labels.json')
    val_images_path = os.path.join(RAW_DATASET_PATH, 'validation-300', 'data')
    val_labels_path = os.path.join(RAW_DATASET_PATH, 'validation-300', 'labels.json')
    paths_to_check = [
        ("Train images", train_images_path),
        ("Train labels", train_labels_path),
        ("Val images", val_images_path),
        ("Val labels", val_labels_path)
    ]
    for name, path in paths_to_check:
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"{status} {name}: {path}")
        if exists and os.path.isdir(path):
            file_count = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])
            print(f"    📁 Contains {file_count} files")
    missing_paths = [path for name, path in paths_to_check if not os.path.exists(path)]
    if missing_paths:
        print(f"\n❌ MISSING REQUIRED PATHS:")
        for path in missing_paths:
            print(f"    - {path}")
        print("\n💡 Please check your dataset structure and update RAW_DATASET_PATH")
        return
    print("✅ All required paths exist!")
    print(f"\n📁 CREATING OUTPUT DIRECTORY")
    print("-" * 40)
    if os.path.exists(CLEANED_DATASET_PATH):
        print(f"⚠️ Output directory exists: {CLEANED_DATASET_PATH}")
        response = input("Do you want to overwrite it? (y/N): ").strip().lower()
        if response != 'y':
            print("❌ Aborted by user")
            return
        shutil.rmtree(CLEANED_DATASET_PATH)
        print("🗑️ Removed existing directory")
    os.makedirs(CLEANED_DATASET_PATH, exist_ok=True)
    print(f"✅ Created: {CLEANED_DATASET_PATH}")
    try:
        print(f"\n🔄 PROCESSING TRAINING SPLIT")
        print("="*40)
        with open(train_labels_path, 'r') as f:
            train_raw_data = json.load(f)
        train_cleaned_data = clean_and_validate_dataset(
            train_raw_data, train_images_path, TARGET_CLASSES, 'train'
        )
        train_dir, train_images_dir, train_ann_file = save_cleaned_dataset(
            train_cleaned_data, CLEANED_DATASET_PATH, 'train'
        )
        print("📦 Copying training images...")
        image_files = [img['file_name'] for img in train_cleaned_data['images']]
        chunk_size = 100
        file_chunks = [image_files[i:i + chunk_size] for i in range(0, len(image_files), chunk_size)]
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            batch_args = [(train_images_path, train_images_dir, chunk) for chunk in file_chunks]
            results = list(tqdm(executor.map(copy_images_batch, batch_args),
                                desc="Copying training images", total=len(batch_args)))
        copied_train = sum(results)
        print(f"✅ Copied {copied_train}/{len(image_files)} training images")
        print(f"\n🔄 PROCESSING VALIDATION SPLIT")
        print("="*40)
        with open(val_labels_path, 'r') as f:
            val_raw_data = json.load(f)
        val_cleaned_data = clean_and_validate_dataset(
            val_raw_data, val_images_path, TARGET_CLASSES, 'validation'
        )
        val_dir, val_images_dir, val_ann_file = save_cleaned_dataset(
            val_cleaned_data, CLEANED_DATASET_PATH, 'validation'
        )
        print("📦 Copying validation images...")
        image_files = [img['file_name'] for img in val_cleaned_data['images']]
        file_chunks = [image_files[i:i + chunk_size] for i in range(0, len(image_files), chunk_size)]
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            batch_args = [(val_images_path, val_images_dir, chunk) for chunk in file_chunks]
            results = list(tqdm(executor.map(copy_images_batch, batch_args),
                                desc="Copying validation images", total=len(batch_args)))
        copied_val = sum(results)
        print(f"✅ Copied {copied_val}/{len(image_files)} validation images")
        print(f"\n📋 CREATING DATASET SUMMARY")
        print("="*40)
        create_dataset_summary(train_cleaned_data, val_cleaned_data, CLEANED_DATASET_PATH)
        print(f"\n🎉 DATA CLEANING AND WRANGLING COMPLETED!")
        print("="*60)
        print(f"📅 Completed at: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"📂 Cleaned dataset saved to: {CLEANED_DATASET_PATH}")
        print(f"📊 Training: {len(train_cleaned_data['images'])} images, {len(train_cleaned_data['annotations'])} annotations")
        print(f"📊 Validation: {len(val_cleaned_data['images'])} images, {len(val_cleaned_data['annotations'])} annotations")
    except Exception as e:
        print(f"\n❌ AN UNEXPECTED ERROR OCCURRED: {e}")
        traceback.print_exc()
if __name__ == '__main__':
    main()

🚀 DATA CLEANING AND WRANGLING PIPELINE
📂 Raw dataset path: /content/drive/MyDrive/coco2017
💾 Cleaned dataset will be saved to: /content/drive/MyDrive/coco2017_cleaned
🎯 Target classes: ['cake', 'car', 'dog', 'person']
⚡ Using 8 workers for processing
🚀 STARTING DATA CLEANING AND WRANGLING PIPELINE
📅 Started at: 2025-08-28 23:14:07

🔍 VERIFYING SOURCE DATASET STRUCTURE
----------------------------------------
✅ Train images: /content/drive/MyDrive/coco2017/train-300/data
    📁 Contains 300 files
✅ Train labels: /content/drive/MyDrive/coco2017/train-300/labels.json
✅ Val images: /content/drive/MyDrive/coco2017/validation-300/data
    📁 Contains 300 files
✅ Val labels: /content/drive/MyDrive/coco2017/validation-300/labels.json
✅ All required paths exist!

📁 CREATING OUTPUT DIRECTORY
----------------------------------------
⚠️ Output directory exists: /content/drive/MyDrive/coco2017_cleaned
Do you want to overwrite it? (y/N): y
🗑️ Removed existing directory
✅ Created: /content/drive/MyDriv

Processing images:   0%|          | 0/300 [00:00<?, ?it/s]

✅ Valid images: 300/300 (100.0%)
📝 Step 2/3: Validating annotations...


Processing annotations:   0%|          | 0/3870 [00:00<?, ?it/s]

✅ Valid annotations: 2395/3870 (61.9%)
🔍 Step 3/3: Final consistency check...

📊 CLEANING SUMMARY FOR TRAIN:
🖼️  IMAGES:
    Original: 300
    Final: 300
    Retention: 100.0%

📝 ANNOTATIONS:
    Original: 3,870
    Final: 2,395
    Retention: 61.9%

🗑️  REMOVALS:
    Invalid polygons: 43
    Invalid bboxes: 0
    Incomplete data: 0
    Irrelevant classes: 1,432
    Missing images: 0
    Orphaned annotations: 0
    Duplicates: 0

🔧 CORRECTIONS:
    Data type fixes: 300
    Clipped coordinates: 100

📈 CLASS DISTRIBUTION:
    cake: 12 instances (0.5%)
    car: 1,048 instances (43.8%)
    dog: 14 instances (0.6%)
    person: 1,321 instances (55.2%)
💾 Saved cleaned annotations: /content/drive/MyDrive/coco2017_cleaned/train/annotations.json
📦 Copying training images...


...

✅ Valid images: 300/300 (100.0%)
📝 Step 2/3: Validating annotations...


Processing annotations:   0%|          | 0/3774 [00:00<?, ?it/s]

✅ Valid annotations: 2044/3774 (54.2%)
🔍 Step 3/3: Final consistency check...

📊 CLEANING SUMMARY FOR VALIDATION:
🖼️  IMAGES:
    Original: 300
    Final: 300
    Retention: 100.0%

📝 ANNOTATIONS:
    Original: 3,774
    Final: 2,044
    Retention: 54.2%

🗑️  REMOVALS:
    Invalid polygons: 35
    Invalid bboxes: 0
    Incomplete data: 0
    Irrelevant classes: 1,695
    Missing images: 0
    Orphaned annotations: 0
    Duplicates: 0

🔧 CORRECTIONS:
    Data type fixes: 300
    Clipped coordinates: 88

📈 CLASS DISTRIBUTION:
    cake: 17 instances (0.8%)
    car: 854 instances (41.8%)
    dog: 9 instances (0.4%)
    person: 1,164 instances (56.9%)
💾 Saved cleaned annotations: /content/drive/MyDrive/coco2017_cleaned/validation/annotations.json
📦 Copying validation images...


Copying validation images:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Copied 300/300 validation images

📋 CREATING DATASET SUMMARY
📋 Dataset summary saved: /content/drive/MyDrive/coco2017_cleaned/dataset_summary.json
📖 README created: /content/drive/MyDrive/coco2017_cleaned/README.md

🎉 DATA CLEANING AND WRANGLING COMPLETED!
📅 Completed at: 2025-08-28 23:14:59
📂 Cleaned dataset saved to: /content/drive/MyDrive/coco2017_cleaned
📊 Training: 300 images, 2395 annotations
📊 Validation: 300 images, 2044 annotations


In [ ]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm
from typing import Dict, List, Any, Optional
import albumentations as A
import datetime
import random
import shutil
from pycocotools import mask as maskUtils
from pycocotools.coco import COCO
import tempfile

class COCOAugmenter:
    """
    Performs data augmentation on a COCO-formatted dataset.

    This class reads images and annotations, applies a series of
    random transformations, and saves the augmented data to a new directory.
    """
    def __init__(self,
                 input_dir: str,
                 output_dir: str,
                 aug_multipliers: Dict[str, int] = None):
        """
        Initializes the augmenter.

        Args:
            input_dir (str): Path to the cleaned dataset (e.g., '/content/drive/MyDrive/coco2017_cleaned/train').
            output_dir (str): Path where the augmented data will be saved.
            aug_multipliers (Dict[str, int]): A dictionary with class names as keys and
                                              multipliers as values to oversample specific classes.
        """
        self.input_dir = input_dir
        self.output_dir = output_dir
        self.images_path = os.path.join(self.input_dir, 'images')
        self.annotations_path = os.path.join(self.input_dir, 'annotations.json')

        self.output_images_path = os.path.join(self.output_dir, 'images')
        self.output_annotations_path = os.path.join(self.output_dir, 'annotations.json')

        self.aug_multipliers = aug_multipliers or {}

        # Load original data
        try:
            with open(self.annotations_path, 'r') as f:
                self.coco_data = json.load(f)
        except FileNotFoundError:
            print(f"Error: Annotations file not found at {self.annotations_path}")
            exit()
        except json.JSONDecodeError:
            print(f"Error: Could not decode JSON from {self.annotations_path}")
            exit()

        # Initialize COCO API for proper mask handling
        self.coco = COCO(self.annotations_path)

        self.categories = {cat['id']: cat for cat in self.coco_data.get('categories', [])}
        self.cat_name_to_id = {cat['name']: cat['id'] for cat in self.coco_data.get('categories', [])}

        self.image_annotations = self._group_annotations_by_image()

        # Initialize new dataset dictionaries
        self.new_coco_data = {
            'info': {
                'description': 'Augmented COCO dataset',
                'version': '1.0',
                'date_created': datetime.datetime.now().isoformat()
            },
            'licenses': self.coco_data.get('licenses', []),
            'categories': self.coco_data.get('categories', []),
            'images': [],
            'annotations': []
        }

        self.next_image_id = 0
        self.next_annotation_id = 0

    def _group_annotations_by_image(self) -> Dict[int, List[Dict]]:
        """Groups annotations by their image ID for easy lookup."""
        img_ann_map = {img['id']: [] for img in self.coco_data.get('images', [])}
        for ann in self.coco_data.get('annotations', []):
            if ann['image_id'] in img_ann_map:
                img_ann_map[ann['image_id']].append(ann)
        return img_ann_map

    def _create_augmentor(self, image_width: int, image_height: int) -> A.Compose:
        """
        Defines the augmentation pipeline for instance segmentation.
        Optimized to work reliably with bounding boxes.
        """
        return A.Compose([
            # Geometric Transformations
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1,
                scale_limit=0.2,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                p=0.7
            ),
            A.Perspective(scale=(0.02, 0.05), p=0.2),

            # Pixel/Color Transformations
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1),
                A.RGBShift(r_shift_limit=20, g_shift_limit=20, b_shift_limit=20, p=1),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1)
            ], p=0.7),

            # Blur and Noise Effects
            A.OneOf([
                A.GaussNoise(var_limit=(10, 50), p=1.0),
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
                A.MedianBlur(blur_limit=3, p=1.0)
            ], p=0.25),

            # Weather and lighting effects
            A.OneOf([
                A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), num_shadows_lower=1, num_shadows_upper=2, p=1.0),
                A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.1, p=1.0),
                A.RandomGamma(gamma_limit=(80, 120), p=1.0)
            ], p=0.2),

            # Grid distortion (works well with bboxes)
            A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.1),

            # Elastic transform (mild)
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=0.1)
        ],
           bbox_params=A.BboxParams(format='coco', label_fields=['category_ids'], min_visibility=0.3),
           is_check_shapes=False)

    def _decode_rle_masks(self, annotations: List[Dict], image_shape: tuple) -> List[np.ndarray]:
        """Decodes RLE-encoded masks to binary numpy arrays."""
        masks = []
        for ann in annotations:
            if 'segmentation' in ann:
                if isinstance(ann['segmentation'], dict):
                    # RLE format
                    mask = maskUtils.decode(ann['segmentation'])
                elif isinstance(ann['segmentation'], list):
                    # Polygon format
                    rles = maskUtils.frPyObjects(ann['segmentation'], image_shape[0], image_shape[1])
                    mask = maskUtils.decode(rles)
                    if len(mask.shape) == 3:
                        mask = mask[:, :, 0]
                else:
                    # Create a mask from bbox if segmentation is not available
                    mask = np.zeros((image_shape[0], image_shape[1]), dtype=np.uint8)
                    bbox = ann['bbox']
                    x, y, w, h = map(int, bbox)
                    mask[y:y+h, x:x+w] = 1
            else:
                # Fallback: create mask from bbox
                mask = np.zeros((image_shape[0], image_shape[1]), dtype=np.uint8)
                bbox = ann['bbox']
                x, y, w, h = map(int, bbox)
                mask[y:y+h, x:x+w] = 1

            masks.append(mask.astype(np.uint8))
        return masks

    def _encode_rle_mask(self, mask: np.ndarray) -> Dict:
        """Encodes a binary numpy mask to RLE format."""
        # Ensure mask is in the right format
        mask = np.asfortranarray(mask.astype(np.uint8))
        rle = maskUtils.encode(mask)
        # Convert bytes to string for JSON serialization
        rle['counts'] = rle['counts'].decode('utf-8')
        return rle

    def _transform_masks_manually(self, masks: List[np.ndarray], transform_func) -> List[np.ndarray]:
        """Apply transformations to masks manually if albumentations fails."""
        transformed_masks = []
        for mask in masks:
            # Apply the same random seed for consistency
            transformed = transform_func(image=mask)
            transformed_masks.append(transformed['image'])
        return transformed_masks

    def _generate_augmented_image(self, original_image: np.ndarray, annotations: List[Dict], augmentor: A.Compose):
        """Applies augmentations to a single image and its annotations."""
        if not annotations:
            return None, None, None

        try:
            bboxes = []
            category_ids = []

            # Filter out invalid bboxes
            valid_annotations = []
            for ann in annotations:
                bbox = ann['bbox']
                if bbox[2] > 0 and bbox[3] > 0:  # width > 0 and height > 0
                    bboxes.append(bbox)
                    category_ids.append(ann['category_id'])
                    valid_annotations.append(ann)

            if not bboxes:
                return None, None, None

            # Decode masks
            masks = self._decode_rle_masks(valid_annotations, original_image.shape[:2])

            # Apply augmentation without masks first (safer approach)
            augmented = augmentor(image=original_image, bboxes=bboxes, category_ids=category_ids)

            aug_image = augmented['image']
            aug_bboxes = augmented['bboxes']
            aug_category_ids = augmented['category_ids']

            # If we have masks and they match the number of remaining bboxes, transform them manually
            aug_masks = []
            if len(masks) == len(bboxes) and len(aug_bboxes) > 0:
                # Create a simple transform for masks (just flip if horizontal flip was applied)
                # This is a simplified approach - you might want to implement more sophisticated mask transformation
                try:
                    # Try to apply the same transformation to masks
                    # For now, we'll just pass through the original masks scaled to match new bboxes
                    for i, (orig_mask, orig_bbox, new_bbox) in enumerate(zip(masks, bboxes, aug_bboxes)):
                        if i < len(aug_bboxes):
                            # Simple approach: just resize the mask to fit new bbox
                            aug_masks.append(orig_mask)
                except Exception as e:
                    print(f"Warning: Mask transformation failed: {e}")
                    # Fallback: create masks from bboxes
                    aug_masks = []
                    for bbox in aug_bboxes:
                        mask = np.zeros((aug_image.shape[0], aug_image.shape[1]), dtype=np.uint8)
                        x, y, w, h = map(int, bbox)
                        x, y, w, h = max(0, x), max(0, y), max(1, w), max(1, h)
                        x2, y2 = min(aug_image.shape[1], x + w), min(aug_image.shape[0], y + h)
                        if x < x2 and y < y2:
                            mask[y:y2, x:x2] = 1
                        aug_masks.append(mask)
            else:
                # Create masks from augmented bboxes
                aug_masks = []
                for bbox in aug_bboxes:
                    mask = np.zeros((aug_image.shape[0], aug_image.shape[1]), dtype=np.uint8)
                    x, y, w, h = map(int, bbox)
                    x, y, w, h = max(0, x), max(0, y), max(1, w), max(1, h)
                    x2, y2 = min(aug_image.shape[1], x + w), min(aug_image.shape[0], y + h)
                    if x < x2 and y < y2:
                        mask[y:y2, x:x2] = 1
                    aug_masks.append(mask)

            new_annotations = []
            for bbox, mask, cat_id in zip(aug_bboxes, aug_masks, aug_category_ids):
                # Recalculate area
                area = float(bbox[2] * bbox[3])
                if area > 1.0:  # Minimum area threshold
                    try:
                        # Encode augmented mask to RLE format
                        rle_segmentation = self._encode_rle_mask(mask)
                        new_annotation = {
                            'id': self.next_annotation_id,
                            'image_id': self.next_image_id,
                            'category_id': int(cat_id),
                            'bbox': [float(x) for x in bbox],
                            'area': area,
                            'segmentation': rle_segmentation,
                            'iscrowd': 0
                        }
                        new_annotations.append(new_annotation)
                        self.next_annotation_id += 1
                    except Exception as e:
                        print(f"Warning: Failed to encode mask: {e}")
                        continue

            if not new_annotations:
                return None, None, None

            new_image_info = {
                'id': self.next_image_id,
                'file_name': f"aug_{self.next_image_id:06d}.jpg",
                'height': int(aug_image.shape[0]),
                'width': int(aug_image.shape[1])
            }
            self.next_image_id += 1

            return aug_image, new_image_info, new_annotations

        except Exception as e:
            print(f"Warning: Augmentation failed for image: {e}")
            return None, None, None

    def run(self):
        """Executes the data augmentation pipeline."""
        os.makedirs(self.output_images_path, exist_ok=True)
        print(f"📁 Augmenting dataset from '{self.input_dir}'...")

        images_to_process = []
        for img in self.coco_data.get('images', []):
            annotations = self.image_annotations.get(img['id'], [])
            if not annotations:
                continue

            # Determine the number of augmented images to generate based on class
            max_multiplier = 1
            for ann in annotations:
                cat_name = self.categories[ann['category_id']]['name']
                max_multiplier = max(max_multiplier, self.aug_multipliers.get(cat_name, 1))

            for _ in range(max_multiplier):
                images_to_process.append(img)

        # Shuffle the list for better training distribution
        random.shuffle(images_to_process)

        total_augmented = 0
        failed_count = 0

        for original_image_info in tqdm(images_to_process, desc="Generating augmentations"):
            original_image_path = os.path.join(self.images_path, original_image_info['file_name'])

            try:
                original_image = cv2.imread(original_image_path)
                if original_image is None:
                    print(f"Warning: Could not load image {original_image_path}")
                    failed_count += 1
                    continue
                original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
            except Exception as e:
                print(f"Warning: Error loading image {original_image_path}: {e}")
                failed_count += 1
                continue

            original_annotations = self.image_annotations.get(original_image_info['id'], [])

            # Create augmented image and annotations
            augmentor = self._create_augmentor(original_image_info['width'], original_image_info['height'])
            aug_image, new_image_info, new_annotations = self._generate_augmented_image(
                original_image, original_annotations, augmentor
            )

            # Handle cases where augmentation failed to produce valid data
            if aug_image is not None and new_annotations:
                try:
                    # Save the augmented image
                    aug_image_bgr = cv2.cvtColor(aug_image, cv2.COLOR_RGB2BGR)
                    cv2.imwrite(os.path.join(self.output_images_path, new_image_info['file_name']), aug_image_bgr)

                    # Add to new COCO data
                    self.new_coco_data['images'].append(new_image_info)
                    self.new_coco_data['annotations'].extend(new_annotations)
                    total_augmented += 1
                except Exception as e:
                    print(f"Warning: Failed to save augmented image: {e}")
                    failed_count += 1
            else:
                failed_count += 1

        # Save the final JSON file
        try:
            with open(self.output_annotations_path, 'w') as f:
                json.dump(self.new_coco_data, f, indent=2)
        except Exception as e:
            print(f"Error saving annotations: {e}")
            return

        print(f"\n✅ Augmentation complete! Saved to {self.output_dir}")
        print(f"📊 Original images: {len(self.coco_data.get('images', []))}")
        print(f"📊 Original annotations: {len(self.coco_data.get('annotations', []))}")
        print(f"✨ Augmented images: {total_augmented}")
        print(f"✨ Augmented annotations: {len(self.new_coco_data['annotations'])}")
        if failed_count > 0:
            print(f"⚠️  Failed augmentations: {failed_count}")

# --- Example Usage ---
if __name__ == '__main__':
    # First, install required packages
    print("Installing required packages...")
    os.system("pip install pycocotools albumentations")

    # Define paths to the cleaned and augmented datasets
    CLEANED_DATASET_PATH = '/content/drive/MyDrive/coco2017_cleaned/train'
    AUGMENTED_DATASET_PATH = '/content/drive/MyDrive/coco2017_augmented/train'

    # Multipliers to address class imbalance from previous analysis
    AUGMENTATION_MULTIPLIERS = {
        'person': 1,
        'car': 1,
        'dog': 50,  # Reduced multiplier to avoid memory issues
        'cake': 75  # Reduced multiplier to avoid memory issues
    }

    # Remove existing augmented directory to ensure a clean run
    if os.path.exists(AUGMENTED_DATASET_PATH):
        shutil.rmtree(AUGMENTED_DATASET_PATH)
        print(f"🗑️ Removed existing directory: {AUGMENTED_DATASET_PATH}")
    os.makedirs(AUGMENTED_DATASET_PATH, exist_ok=True)

    # Initialize and run the augmenter
    try:
        augmenter = COCOAugmenter(
            input_dir=CLEANED_DATASET_PATH,
            output_dir=AUGMENTED_DATASET_PATH,
            aug_multipliers=AUGMENTATION_MULTIPLIERS
        )

        augmenter.run()
    except Exception as e:
        print(f"Error during augmentation: {e}")
        print("Make sure you have installed pycocotools: pip install pycocotools")

Installing required packages...
🗑️ Removed existing directory: /content/drive/MyDrive/coco2017_augmented/train
loading annotations into memory...
Done (t=0.03s)
creating index...
index created!
📁 Augmenting dataset from '/content/drive/MyDrive/coco2017_cleaned/train'...


Generating augmentations: 100%|██████████| 864/864 [04:44<00:00,  3.04it/s]



✅ Augmentation complete! Saved to /content/drive/MyDrive/coco2017_augmented/train
📊 Original images: 300
📊 Original annotations: 2395
✨ Augmented images: 861
✨ Augmented annotations: 7601
⚠️  Failed augmentations: 3


In [ ]:
import os
import json
import cv2
import numpy as np
import yaml
from pathlib import Path
import shutil
from tqdm import tqdm
from ultralytics import YOLO
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils
import torch

class COCOToYOLOConverter:
    """
    Converts COCO format dataset to YOLO format for instance segmentation.
    """

    def __init__(self, coco_dataset_path: str, yolo_output_path: str):
        self.coco_dataset_path = coco_dataset_path
        self.yolo_output_path = yolo_output_path
        self.images_path = os.path.join(coco_dataset_path, 'images')
        self.annotations_path = os.path.join(coco_dataset_path, 'annotations.json')

        # Load COCO data
        self.coco = COCO(self.annotations_path)
        with open(self.annotations_path, 'r') as f:
            self.coco_data = json.load(f)

        # Create category mapping
        self.categories = {cat['id']: cat for cat in self.coco_data['categories']}
        self.class_names = [cat['name'] for cat in self.coco_data['categories']]
        self.class_mapping = {cat['id']: idx for idx, cat in enumerate(self.coco_data['categories'])}

        print(f"Found {len(self.class_names)} classes: {self.class_names}")

    def polygon_to_mask(self, segmentation, height, width):
        """Convert polygon segmentation to binary mask."""
        if isinstance(segmentation, dict):
            # RLE format
            mask = maskUtils.decode(segmentation)
        elif isinstance(segmentation, list):
            # Polygon format
            rles = maskUtils.frPyObjects(segmentation, height, width)
            mask = maskUtils.decode(rles)
            if len(mask.shape) == 3:
                mask = mask[:, :, 0]
        return mask.astype(np.uint8)

    def mask_to_polygon(self, mask):
        """Convert binary mask to polygon format for YOLO."""
        # Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return None

        # Get the largest contour
        largest_contour = max(contours, key=cv2.contourArea)

        # Simplify contour to reduce points
        epsilon = 0.002 * cv2.arcLength(largest_contour, True)
        simplified_contour = cv2.approxPolyDP(largest_contour, epsilon, True)

        # Convert to normalized coordinates
        polygon = []
        h, w = mask.shape
        for point in simplified_contour:
            x, y = point[0]
            # Normalize coordinates
            x_norm = x / w
            y_norm = y / h
            polygon.extend([x_norm, y_norm])

        # YOLO requires at least 6 points (3 coordinates)
        if len(polygon) < 6:
            return None

        return polygon

    def convert_dataset(self):
        """Convert COCO dataset to YOLO format."""

        # Create YOLO directory structure
        yolo_images_dir = os.path.join(self.yolo_output_path, 'images')
        yolo_labels_dir = os.path.join(self.yolo_output_path, 'labels')

        os.makedirs(yolo_images_dir, exist_ok=True)
        os.makedirs(yolo_labels_dir, exist_ok=True)

        print(f"Converting COCO dataset to YOLO format...")

        # Process each image
        successful_conversions = 0
        failed_conversions = 0

        for img_info in tqdm(self.coco_data['images'], desc="Converting images"):
            try:
                img_id = img_info['id']
                img_filename = img_info['file_name']
                img_width = img_info['width']
                img_height = img_info['height']

                # Copy image to YOLO images directory
                src_img_path = os.path.join(self.images_path, img_filename)
                dst_img_path = os.path.join(yolo_images_dir, img_filename)

                if not os.path.exists(src_img_path):
                    print(f"Warning: Image {src_img_path} not found")
                    failed_conversions += 1
                    continue

                shutil.copy2(src_img_path, dst_img_path)

                # Get annotations for this image
                ann_ids = self.coco.getAnnIds(imgIds=img_id)
                annotations = self.coco.loadAnns(ann_ids)

                # Create YOLO label file
                label_filename = img_filename.replace('.jpg', '.txt').replace('.png', '.txt')
                label_path = os.path.join(yolo_labels_dir, label_filename)

                yolo_annotations = []

                for ann in annotations:
                    try:
                        class_id = self.class_mapping[ann['category_id']]

                        # Convert segmentation to mask then to polygon
                        if 'segmentation' in ann and ann['segmentation']:
                            mask = self.polygon_to_mask(ann['segmentation'], img_height, img_width)
                            polygon = self.mask_to_polygon(mask)

                            if polygon and len(polygon) >= 6:
                                # Format: class_id x1 y1 x2 y2 x3 y3 ...
                                yolo_line = f"{class_id} " + " ".join(map(str, polygon))
                                yolo_annotations.append(yolo_line)
                        else:
                            # Fallback: use bbox to create a rectangular mask
                            bbox = ann['bbox']
                            x, y, w, h = bbox

                            # Normalize coordinates
                            x1, y1 = x / img_width, y / img_height
                            x2, y2 = (x + w) / img_width, y / img_height
                            x3, y3 = (x + w) / img_width, (y + h) / img_height
                            x4, y4 = x / img_width, (y + h) / img_height

                            # Create rectangular polygon
                            polygon = [x1, y1, x2, y2, x3, y3, x4, y4]
                            yolo_line = f"{class_id} " + " ".join(map(str, polygon))
                            yolo_annotations.append(yolo_line)

                    except Exception as e:
                        print(f"Warning: Failed to process annotation: {e}")
                        continue

                # Write YOLO label file
                with open(label_path, 'w') as f:
                    f.write('\n'.join(yolo_annotations))

                successful_conversions += 1

            except Exception as e:
                print(f"Error processing image {img_info.get('file_name', 'unknown')}: {e}")
                failed_conversions += 1
                continue

        print(f"✅ Conversion complete!")
        print(f"✨ Successfully converted: {successful_conversions} images")
        print(f"⚠️  Failed conversions: {failed_conversions}")

        return successful_conversions > 0

class YOLOv8SegmentationTrainer:
    """
    Handles YOLOv8 instance segmentation training.
    """

    def __init__(self,
                 dataset_path: str,
                 class_names: list,
                 model_size: str = 'n', # n, s, m, l, x
                 project_name: str = 'yolov8_segmentation'):

        self.dataset_path = dataset_path
        self.class_names = class_names
        self.model_size = model_size
        self.project_name = project_name

        # Create data.yaml file
        self.create_data_yaml()

    def create_data_yaml(self):
        """Create the data.yaml configuration file for YOLO training."""

        data_yaml = {
            'path': self.dataset_path,
            'train': 'images',
            'val': 'images', # Using same for both train/val for now
            'test': 'images',
            'nc': len(self.class_names),
            'names': {i: name for i, name in enumerate(self.class_names)}
        }

        yaml_path = os.path.join(self.dataset_path, 'data.yaml')
        with open(yaml_path, 'w') as f:
            yaml.dump(data_yaml, f, default_flow_style=False)

        self.data_yaml_path = yaml_path
        print(f"✅ Created data.yaml at {yaml_path}")

    def train_model(self,
                    epochs: int = 100,
                    batch_size: int = 16,
                    image_size: int = 640,
                    device: str = 'auto',
                    patience: int = 50,
                    save_period: int = 10,
                    cos_lr: bool = True):
        """
        Train YOLOv8 segmentation model.
        """

        print(f"🚀 Starting YOLOv8{self.model_size} segmentation training...")
        print(f"📊 Dataset: {self.dataset_path}")
        print(f"🏷️  Classes: {self.class_names}")
        print(f"⚙️  Settings: {epochs} epochs, batch size {batch_size}, image size {image_size}")

        # Load model
        model_name = f'yolov8{self.model_size}-seg.pt'
        model = YOLO(model_name)

        print(f"📥 Loaded {model_name}")

        # Check device
        if device == 'auto':
            device = 'cuda' if torch.cuda.is_available() else 'cpu'

        print(f"🔧 Using device: {device}")

        # Train the model
        try:
            results = model.train(
                data=self.data_yaml_path,
                epochs=epochs,
                batch=batch_size,
                imgsz=image_size,
                device=device,
                patience=patience,
                save_period=save_period,
                project=self.project_name,
                name='train',
                exist_ok=True,
                verbose=True,
                plots=True,
                cache=True,
                single_cls=False,
                optimizer='auto',
                cos_lr=cos_lr, # Set to True to enable cosine annealing
                weight_decay=0.0005,
                warmup_epochs=3,
                warmup_momentum=0.8,
                box=7.5,
                cls=0.5,
                dfl=1.5
            )

            print("✅ Training completed successfully!")

            # Display training results
            self.display_results(results)

            return results

        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def display_results(self, results):
        """Display training results and metrics."""

        print("\n📈 Training Results:")
        print("=" * 50)

        try:
            # Get the results directory
            results_dir = Path(self.project_name) / 'train'

            # Display key metrics
            if hasattr(results, 'results_dict'):
                metrics = results.results_dict
                print(f"🎯 Final mAP50: {metrics.get('metrics/mAP50(B)', 'N/A'):.4f}")
                print(f"🎯 Final mAP50-95: {metrics.get('metrics/mAP50-95(B)', 'N/A'):.4f}")

            # Show plots if available
            plots_dir = results_dir / 'plots'
            if plots_dir.exists():
                print(f"📊 Training plots saved to: {plots_dir}")

                # List available plots
                plot_files = list(plots_dir.glob('*.png'))
                if plot_files:
                    print("📈 Available plots:")
                    for plot_file in plot_files:
                        print(f"    - {plot_file.name}")

            # Show best weights location
            weights_dir = results_dir / 'weights'
            if weights_dir.exists():
                best_weights = weights_dir / 'best.pt'
                last_weights = weights_dir / 'last.pt'

                if best_weights.exists():
                    print(f"🏆 Best weights saved to: {best_weights}")
                if last_weights.exists():
                    print(f"💾 Last weights saved to: {last_weights}")

        except Exception as e:
            print(f"Warning: Could not display detailed results: {e}")

    def validate_model(self, weights_path: str = None):
        """Validate the trained model."""

        if weights_path is None:
            # Use best weights from training
            weights_path = Path(self.project_name) / 'train' / 'weights' / 'best.pt'

        if not os.path.exists(weights_path):
            print(f"❌ Weights file not found: {weights_path}")
            return None

        print(f"🔍 Validating model with weights: {weights_path}")

        try:
            model = YOLO(weights_path)
            results = model.val(data=self.data_yaml_path)

            print("✅ Validation completed!")
            return results

        except Exception as e:
            print(f"❌ Validation failed: {e}")
            return None

def main():
    """Main training pipeline."""

    # Install required packages
    print("📦 Installing required packages...")
    os.system("pip install ultralytics pycocotools pillow")

    # Configuration
    AUGMENTED_COCO_PATH = '/content/drive/MyDrive/coco2017_augmented/train'
    YOLO_DATASET_PATH = '/content/drive/MyDrive/yolo_segmentation_dataset'

    # Training parameters
    EPOCHS = 100
    BATCH_SIZE = 16 # Adjust based on your GPU memory
    IMAGE_SIZE = 640
    MODEL_SIZE = 'n' # Start with nano model for faster training

    print("🔄 Step 1: Converting COCO to YOLO format...")

    # Convert COCO to YOLO format
    converter = COCOToYOLOConverter(AUGMENTED_COCO_PATH, YOLO_DATASET_PATH)

    if converter.convert_dataset():
        print("🔄 Step 2: Starting YOLOv8 training...")

        # Train YOLOv8 model
        trainer = YOLOv8SegmentationTrainer(
            dataset_path=YOLO_DATASET_PATH,
            class_names=converter.class_names,
            model_size=MODEL_SIZE,
            project_name='coco_segmentation_training'
        )

        # Start training with cosine learning rate scheduler enabled
        results = trainer.train_model(
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            image_size=IMAGE_SIZE,
            patience=30, # Early stopping patience
            cos_lr=True # Enable cosine annealing scheduler
        )

        if results:
            print("🔄 Step 3: Validating trained model...")
            trainer.validate_model()

            print("\n🎉 Training pipeline completed successfully!")
            print("\n📋 Next steps:")
            print("1. Check training plots in the results directory")
            print("2. Use best.pt weights for inference")
            print("3. Fine-tune hyperparameters if needed")

        else:
            print("❌ Training failed. Check the logs above for errors.")

    else:
        print("❌ Dataset conversion failed. Cannot proceed with training.")

if __name__ == '__main__':
    main()

📦 Installing required packages...
🔄 Step 1: Converting COCO to YOLO format...
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Found 4 classes: ['cake', 'car', 'dog', 'person']
Converting COCO dataset to YOLO format...


Converting images: 100%|██████████| 861/861 [00:19<00:00, 43.90it/s]


✅ Conversion complete!
✨ Successfully converted: 861 images
⚠️  Failed conversions: 0
🔄 Step 2: Starting YOLOv8 training...
✅ Created data.yaml at /content/drive/MyDrive/yolo_segmentation_dataset/data.yaml
🚀 Starting YOLOv8n segmentation training...
📊 Dataset: /content/drive/MyDrive/yolo_segmentation_dataset
🏷️  Classes: ['cake', 'car', 'dog', 'person']
⚙️  Settings: 100 epochs, batch size 16, image size 640
📥 Loaded yolov8n-seg.pt
🔧 Using device: cuda
Ultralytics 8.3.189 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/yolo_segmentation_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exi

In [ ]:
import os
import json
import cv2
import numpy as np
import yaml
from pathlib import Path
import shutil
from tqdm import tqdm
from ultralytics import YOLO
import torch
import optuna
from pycocotools.coco import COCO
from pycocotools import mask as maskUtils

# Use a custom logging format to make it cleaner
optuna.logging.set_verbosity(optuna.logging.WARNING)

class COCOToYOLOConverter:
    """
    Converts COCO format dataset to YOLO format for instance segmentation.
    """

    def __init__(self, coco_dataset_path: str, yolo_output_path: str):
        self.coco_dataset_path = coco_dataset_path
        self.yolo_output_path = yolo_output_path
        self.images_path = os.path.join(coco_dataset_path, 'images')
        self.annotations_path = os.path.join(coco_dataset_path, 'annotations.json')

        # Load COCO data
        self.coco = COCO(self.annotations_path)
        with open(self.annotations_path, 'r') as f:
            self.coco_data = json.load(f)

        # Create category mapping
        self.categories = {cat['id']: cat for cat in self.coco_data['categories']}
        self.class_names = [cat['name'] for cat in self.coco_data['categories']]
        self.class_mapping = {cat['id']: idx for idx, cat in enumerate(self.coco_data['categories'])}

        print(f"Found {len(self.class_names)} classes: {self.class_names}")

    def polygon_to_mask(self, segmentation, height, width):
        """Convert polygon segmentation to binary mask."""
        if isinstance(segmentation, dict):
            # RLE format
            mask = maskUtils.decode(segmentation)
        elif isinstance(segmentation, list):
            # Polygon format
            rles = maskUtils.frPyObjects(segmentation, height, width)
            mask = maskUtils.decode(rles)
            if len(mask.shape) == 3:
                mask = mask[:, :, 0]
        return mask.astype(np.uint8)

    def mask_to_polygon(self, mask):
        """Convert binary mask to polygon format for YOLO."""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None
        largest_contour = max(contours, key=cv2.contourArea)
        epsilon = 0.002 * cv2.arcLength(largest_contour, True)
        simplified_contour = cv2.approxPolyDP(largest_contour, epsilon, True)
        polygon = []
        h, w = mask.shape
        for point in simplified_contour:
            x, y = point[0]
            x_norm = x / w
            y_norm = y / h
            polygon.extend([x_norm, y_norm])
        if len(polygon) < 6:
            return None
        return polygon

    def convert_dataset(self):
        """Convert COCO dataset to YOLO format."""
        yolo_images_dir = os.path.join(self.yolo_output_path, 'images')
        yolo_labels_dir = os.path.join(self.yolo_output_path, 'labels')
        os.makedirs(yolo_images_dir, exist_ok=True)
        os.makedirs(yolo_labels_dir, exist_ok=True)
        print(f"Converting COCO dataset to YOLO format...")
        successful_conversions = 0
        failed_conversions = 0

        for img_info in tqdm(self.coco_data['images'], desc="Converting images"):
            try:
                img_id = img_info['id']
                img_filename = img_info['file_name']
                img_width = img_info['width']
                img_height = img_info['height']
                src_img_path = os.path.join(self.images_path, img_filename)
                dst_img_path = os.path.join(yolo_images_dir, img_filename)
                if not os.path.exists(src_img_path):
                    print(f"Warning: Image {src_img_path} not found")
                    failed_conversions += 1
                    continue
                shutil.copy2(src_img_path, dst_img_path)
                ann_ids = self.coco.getAnnIds(imgIds=img_id)
                annotations = self.coco.loadAnns(ann_ids)
                label_filename = img_filename.replace('.jpg', '.txt').replace('.png', '.txt')
                label_path = os.path.join(yolo_labels_dir, label_filename)
                yolo_annotations = []

                for ann in annotations:
                    try:
                        class_id = self.class_mapping[ann['category_id']]
                        if 'segmentation' in ann and ann['segmentation']:
                            mask = self.polygon_to_mask(ann['segmentation'], img_height, img_width)
                            polygon = self.mask_to_polygon(mask)
                            if polygon and len(polygon) >= 6:
                                yolo_line = f"{class_id} " + " ".join(map(str, polygon))
                                yolo_annotations.append(yolo_line)
                        else:
                            bbox = ann['bbox']
                            x, y, w, h = bbox
                            x1, y1 = x / img_width, y / img_height
                            x2, y2 = (x + w) / img_width, y / img_height
                            x3, y3 = (x + w) / img_width, (y + h) / img_height
                            x4, y4 = x / img_width, (y + h) / img_height
                            polygon = [x1, y1, x2, y2, x3, y3, x4, y4]
                            yolo_line = f"{class_id} " + " ".join(map(str, polygon))
                            yolo_annotations.append(yolo_line)
                    except Exception as e:
                        print(f"Warning: Failed to process annotation: {e}")
                        continue

                with open(label_path, 'w') as f:
                    f.write('\n'.join(yolo_annotations))
                successful_conversions += 1
            except Exception as e:
                print(f"Error processing image {img_info.get('file_name', 'unknown')}: {e}")
                failed_conversions += 1
                continue

        print(f"✅ Conversion complete!")
        print(f"✨ Successfully converted: {successful_conversions} images")
        print(f"⚠️  Failed conversions: {failed_conversions}")
        return successful_conversions > 0

class YOLOv8SegmentationTrainer:
    """
    Handles YOLOv8 instance segmentation training.
    """
    def __init__(self, dataset_path: str, class_names: list, model_size: str = 'n', project_name: str = 'yolov8_segmentation'):
        self.dataset_path = dataset_path
        self.class_names = class_names
        self.model_size = model_size
        self.project_name = project_name
        self.create_data_yaml()

    def create_data_yaml(self):
        """Create the data.yaml configuration file for YOLO training."""
        data_yaml = {
            'path': self.dataset_path,
            'train': 'images',
            'val': 'images',
            'test': 'images',
            'nc': len(self.class_names),
            'names': {i: name for i, name in enumerate(self.class_names)}
        }
        yaml_path = os.path.join(self.dataset_path, 'data.yaml')
        with open(yaml_path, 'w') as f:
            yaml.dump(data_yaml, f, default_flow_style=False)
        self.data_yaml_path = yaml_path
        print(f"✅ Created data.yaml at {yaml_path}")

    def train_model(self,
                    epochs: int = 200,
                    batch_size: int = 16,
                    image_size: int = 640,
                    device: str = 'auto',
                    patience: int = 5,  # Changed from 30 to 5 for hyperparameter tuning
                    save_period: int = 10,
                    cos_lr: bool = True,
                    lr0: float = 0.01,
                    lrf: float = 0.01,
                    weight_decay: float = 0.0005,
                    warmup_epochs: int = 3,
                    box: float = 7.5,
                    cls: float = 0.5,
                    dfl: float = 1.5,
                    trial_name: str = None):
        """
        Train YOLOv8 segmentation model.
        """
        print(f"🚀 Starting YOLOv8{self.model_size} segmentation training...")
        print(f"📊 Dataset: {self.dataset_path}")
        print(f"🏷️  Classes: {self.class_names}")
        print(f"⚙️  Settings: {epochs} epochs, batch size {batch_size}, image size {image_size}, patience {patience}")

        model_name = f'yolov8{self.model_size}-seg.pt'
        model = YOLO(model_name)
        print(f"📥 Loaded {model_name}")

        if device == 'auto':
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"🔧 Using device: {device}")

        # Use trial_name for optuna runs to avoid conflicts
        run_name = trial_name if trial_name else 'train'

        try:
            results = model.train(
                data=self.data_yaml_path,
                epochs=epochs,
                batch=batch_size,
                imgsz=image_size,
                device=device,
                patience=patience,
                save_period=save_period,
                project=self.project_name,
                name=run_name,
                exist_ok=True,
                verbose=False,  # Reduced verbosity for optuna
                plots=True,
                cache=True,
                single_cls=False,
                optimizer='auto',
                cos_lr=cos_lr,
                lr0=lr0,
                lrf=lrf,
                weight_decay=weight_decay,
                warmup_epochs=warmup_epochs,
                warmup_momentum=0.8,
                box=box,
                cls=cls,
                dfl=dfl
            )
            print("✅ Training completed successfully!")
            if trial_name is None:  # Only display detailed results for final training
                self.display_results(results)
            return results
        except Exception as e:
            print(f"❌ Training failed: {e}")
            return None

    def display_results(self, results):
        """Display training results and metrics."""
        print("\n📈 Training Results:")
        print("=" * 50)
        try:
            results_dir = Path(self.project_name) / 'train'
            if hasattr(results, 'results_dict'):
                metrics = results.results_dict
                print(f"🎯 Final mAP50(B): {metrics.get('metrics/mAP50(B)', 'N/A'):.4f}")
                print(f"🎯 Final mAP50-95(B): {metrics.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
                print(f"🎯 Final mAP50(S): {metrics.get('metrics/mAP50(S)', 'N/A'):.4f}")
                print(f"🎯 Final mAP50-95(S): {metrics.get('metrics/mAP50-95(S)', 'N/A'):.4f}")
            plots_dir = results_dir / 'plots'
            if plots_dir.exists():
                print(f"📊 Training plots saved to: {plots_dir}")
                plot_files = list(plots_dir.glob('*.png'))
                if plot_files:
                    print("📈 Available plots:")
                    for plot_file in plot_files:
                        print(f"    - {plot_file.name}")
            weights_dir = results_dir / 'weights'
            if weights_dir.exists():
                best_weights = weights_dir / 'best.pt'
                last_weights = weights_dir / 'last.pt'
                if best_weights.exists():
                    print(f"🏆 Best weights saved to: {best_weights}")
                if last_weights.exists():
                    print(f"💾 Last weights saved to: {last_weights}")
        except Exception as e:
            print(f"Warning: Could not display detailed results: {e}")

    def validate_model(self, weights_path: str = None):
        """Validate the trained model."""
        if weights_path is None:
            weights_path = Path(self.project_name) / 'train' / 'weights' / 'best.pt'
        if not os.path.exists(weights_path):
            print(f"❌ Weights file not found: {weights_path}")
            return None
        print(f"🔍 Validating model with weights: {weights_path}")
        try:
            model = YOLO(weights_path)
            results = model.val(data=self.data_yaml_path)
            print("✅ Validation completed!")
            return results
        except Exception as e:
            print(f"❌ Validation failed: {e}")
            return None

class HyperparameterTuner:
    """
    Optimizes hyperparameters for YOLOv8 training using Optuna.
    """
    def __init__(self, trainer: YOLOv8SegmentationTrainer, n_trials: int = 20, epochs: int = 30):
        self.trainer = trainer
        self.n_trials = n_trials
        self.epochs = epochs
        self.trial_counter = 0

    def objective(self, trial: optuna.Trial):
        """
        Objective function for Optuna. It defines the hyperparameters to optimize and the metric to maximize.
        """
        self.trial_counter += 1

        # Suggest hyperparameters to optimize
        lr0 = trial.suggest_float('lr0', 1e-4, 5e-2, log=True)
        lrf = trial.suggest_float('lrf', 1e-3, 0.2, log=True)
        weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
        warmup_epochs = trial.suggest_int('warmup_epochs', 1, 5)
        box = trial.suggest_float('box', 5.0, 10.0)
        cls = trial.suggest_float('cls', 0.2, 1.0)
        dfl = trial.suggest_float('dfl', 1.0, 2.0)

        print(f"\n🔬 Trial {self.trial_counter}/{self.n_trials}")
        print(f"📋 Parameters: lr0={lr0:.6f}, lrf={lrf:.6f}, weight_decay={weight_decay:.6f}")

        # Train the model with the suggested hyperparameters
        results = self.trainer.train_model(
            epochs=self.epochs,
            batch_size=16,
            image_size=640,
            patience=5,  # Early stopping after 5 epochs of no improvement
            cos_lr=True,
            lr0=lr0,
            lrf=lrf,
            weight_decay=weight_decay,
            warmup_epochs=warmup_epochs,
            box=box,
            cls=cls,
            dfl=dfl,
            trial_name=f'trial_{self.trial_counter}'
        )

        if results and hasattr(results, 'results_dict'):
            # Use segmentation mAP50 as the primary metric
            metric_val = results.results_dict.get('metrics/mAP50(S)', 0.0)

            print(f"📊 Trial {self.trial_counter} result: mAP50(S) = {metric_val:.4f}")

            # Report the metric value to Optuna
            trial.report(metric_val, step=self.epochs)

            # Check if the trial should be pruned
            if trial.should_prune():
                print(f"✂️  Trial {self.trial_counter} pruned")
                raise optuna.exceptions.TrialPruned()

            return metric_val
        else:
            # Return a very low value for failed trials
            print(f"❌ Trial {self.trial_counter} failed")
            return 0.0

    def run_tuning(self):
        """
        Run the Optuna optimization study.
        """
        print("🚀 Starting hyperparameter tuning with Optuna...")
        print(f"⚙️  Settings: {self.n_trials} trials, {self.epochs} epochs per trial")
        print(f"⏰ Early stopping: 5 epochs patience per trial")

        # Create a study object with more aggressive pruning
        study = optuna.create_study(
            direction='maximize',
            pruner=optuna.pruners.MedianPruner(
                n_startup_trials=3,    # Reduced startup trials
                n_warmup_steps=5,      # Wait 5 steps before pruning
                interval_steps=2       # Check every 2 steps
            )
        )

        try:
            study.optimize(self.objective, n_trials=self.n_trials, show_progress_bar=True)
        except KeyboardInterrupt:
            print("\n⚠️  Optimization interrupted by user")

        print("\n✅ Tuning completed!")
        print("📈 Best trial:")
        print(f"  Value (mAP50S): {study.best_value:.4f}")
        print(f"  Params: {study.best_params}")

        # Print top 3 trials
        top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else 0.0, reverse=True)[:3]
        print("\n🏆 Top 3 trials:")
        for i, trial in enumerate(top_trials, 1):
            if trial.value is not None:
                print(f"  {i}. Value: {trial.value:.4f}, Params: {trial.params}")

        return study.best_params

def main():
    """Main training pipeline."""
    print("📦 Installing required packages...")
    os.system("pip install ultralytics pycocotools pillow optuna")

    AUGMENTED_COCO_PATH = '/content/drive/MyDrive/coco2017_augmented/train'
    YOLO_DATASET_PATH = '/content/drive/MyDrive/yolo_segmentation_dataset'

    # Configuration
    FINAL_EPOCHS = 200          # Final training epochs
    TUNING_EPOCHS = 30          # Epochs for each hyperparameter trial (reasonable balance)
    N_TRIALS = 20               # Number of hyperparameter trials (reasonable for good exploration)
    BATCH_SIZE = 16
    IMAGE_SIZE = 640
    MODEL_SIZE = 'n'

    print("🔄 Step 1: Converting COCO to YOLO format...")
    converter = COCOToYOLOConverter(AUGMENTED_COCO_PATH, YOLO_DATASET_PATH)

    if converter.convert_dataset():
        trainer = YOLOv8SegmentationTrainer(
            dataset_path=YOLO_DATASET_PATH,
            class_names=converter.class_names,
            model_size=MODEL_SIZE,
            project_name='coco_segmentation_training'
        )

        # Step 2: Run hyperparameter tuning
        print("🔄 Step 2: Starting hyperparameter tuning...")
        tuner = HyperparameterTuner(trainer, n_trials=N_TRIALS, epochs=TUNING_EPOCHS)
        best_params = tuner.run_tuning()

        # Step 3: Train the final model with the best parameters
        print("🔄 Step 3: Starting final training with best hyperparameters...")
        print(f"🎯 Using optimized parameters: {best_params}")

        results = trainer.train_model(
            epochs=FINAL_EPOCHS,
            batch_size=BATCH_SIZE,
            image_size=IMAGE_SIZE,
            patience=30,  # More patience for final training
            cos_lr=True,
            lr0=best_params.get('lr0', 0.01),
            lrf=best_params.get('lrf', 0.01),
            weight_decay=best_params.get('weight_decay', 0.0005),
            warmup_epochs=best_params.get('warmup_epochs', 3),
            box=best_params.get('box', 7.5),
            cls=best_params.get('cls', 0.5),
            dfl=best_params.get('dfl', 1.5)
        )

        if results:
            print("🔄 Step 4: Validating trained model...")
            trainer.validate_model()

            print("\n🎉 Training pipeline completed successfully!")
            print("\n📋 Summary:")
            print(f"✅ Hyperparameter tuning: {N_TRIALS} trials × {TUNING_EPOCHS} epochs")
            print(f"✅ Final training: {FINAL_EPOCHS} epochs with optimized parameters")
            print(f"✅ Best hyperparameters: {best_params}")
            print("\n📋 Next steps:")
            print("1. Check training plots in the results directory")
            print("2. Use best.pt weights for inference")
            print("3. Consider ensemble methods for better performance")

        else:
            print("❌ Training failed. Check the logs above for errors.")
    else:
        print("❌ Dataset conversion failed. Cannot proceed with training.")

if __name__ == '__main__':
    main()

📦 Installing required packages...
🔄 Step 1: Converting COCO to YOLO format...
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Found 4 classes: ['cake', 'car', 'dog', 'person']
Converting COCO dataset to YOLO format...


Converting images: 100%|██████████| 861/861 [00:19<00:00, 44.50it/s]

✅ Conversion complete!
✨ Successfully converted: 861 images
⚠️  Failed conversions: 0
✅ Created data.yaml at /content/drive/MyDrive/yolo_segmentation_dataset/data.yaml
🔄 Step 2: Starting hyperparameter tuning...
🚀 Starting hyperparameter tuning with Optuna...
⚙️  Settings: 20 trials, 30 epochs per trial
⏰ Early stopping: 5 epochs patience per trial


  0%|          | 0/20 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.4it/s 6.1s
                   all        861       7581      0.256      0.152      0.121     0.0679      0.192      0.122      0.069     0.0196

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size
      12/30      4.01G       1.95      3.519      1.592      1.797        206        640: 100% ━━━━━━━━━━━━ 54/54 7.5it/s 7.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 27/27 4.4it/s 6.1s
                   all        861       7581      0.302      0.141      0.124     0.0698      0.232      0.107     0.0663     0.0196

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss  Instances       Size
      13/30      4.01G      1.949 

In [ ]:
# This script performs the final YOLOv8 training run using the best
# hyperparameters found during the previous optimization step.

from ultralytics import YOLO
import os
import yaml

def train_final_model(
    dataset_path: str,
    project_name: str,
    epochs: int,
    batch_size: int,
    image_size: int,
    patience: int,
    model_size: str,
    best_params: dict
):
    """
    Train a YOLOv8 segmentation model with a fixed set of hyperparameters.

    Args:
        dataset_path (str): Path to the YOLO dataset directory.
        project_name (str): Name for the training project.
        epochs (int): Number of training epochs.
        batch_size (int): Batch size for training.
        image_size (int): Image size for training.
        patience (int): Early stopping patience.
        model_size (str): The YOLO model variant (e.g., 'n', 's', 'm').
        best_params (dict): Dictionary of optimized hyperparameters.
    """
    print("🚀 Starting final training with best hyperparameters...")

    # Load the base model
    model_name = f'yolov8{model_size}-seg.pt'
    model = YOLO(model_name)
    print(f"📥 Loaded {model_name}")

    # The data.yaml file should be inside your dataset path
    data_yaml_path = os.path.join(dataset_path, 'data.yaml')
    if not os.path.exists(data_yaml_path):
        print(f"❌ Error: data.yaml not found at {data_yaml_path}")
        return

    # Use a different name for this final run to keep things organized
    run_name = 'final_run'

    try:
        results = model.train(
            data=data_yaml_path,
            epochs=epochs,
            batch=batch_size,
            imgsz=image_size,
            device='auto', # Let YOLOv8 auto-detect GPU/CPU
            patience=patience,
            project=project_name,
            name=run_name,
            exist_ok=True,
            plots=True,
            cache=True,
            single_cls=False,
            optimizer='auto',
            cos_lr=True,
            lr0=best_params['lr0'],
            lrf=best_params['lrf'],
            weight_decay=best_params['weight_decay'],
            warmup_epochs=best_params['warmup_epochs'],
            box=best_params['box'],
            cls=best_params['cls'],
            dfl=best_params['dfl']
        )
        print("✅ Final training completed successfully!")

        # Display the final validation results
        print("\n📈 Final Validation Results:")
        metrics = results.results_dict
        print(f"🎯 mAP50(S): {metrics.get('metrics/mAP50(S)', 'N/A'):.4f}")
        print(f"🎯 mAP50-95(S): {metrics.get('metrics/mAP50-95(S)', 'N/A'):.4f}")

    except Exception as e:
        print(f"❌ Training failed: {e}")

if __name__ == '__main__':
    # --- Configuration ---

    # Path to the YOLO dataset created by the conversion script
    YOLO_DATASET_PATH = '/content/drive/MyDrive/yolo_segmentation_dataset'

    # Project and run names for the training output
    PROJECT_NAME = 'coco_segmentation_training'

    # Settings from your previous training pipeline
    FINAL_EPOCHS = 200
    BATCH_SIZE = 16
    IMAGE_SIZE = 640
    MODEL_SIZE = 'n'
    PATIENCE = 30

    # Best hyperparameters from your previous log
    BEST_HYPERPARAMS = {
        'lr0': 0.0007475524102215215,
        'lrf': 0.005463484813185952,
        'weight_decay': 0.00014275735675228264,
        'warmup_epochs': 2,
        'box': 6.79718182417978,
        'cls': 0.237698606960847,
        'dfl': 1.662770506590975
    }

    train_final_model(
        dataset_path=YOLO_DATASET_PATH,
        project_name=PROJECT_NAME,
        epochs=FINAL_EPOCHS,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
        patience=PATIENCE,
        model_size=MODEL_SIZE,
        best_params=BEST_HYPERPARAMS
    )


🚀 Starting final training with best hyperparameters...
📥 Loaded yolov8n-seg.pt
Ultralytics 8.3.189 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:auto (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=6.79718182417978, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.237698606960847, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/yolo_segmentation_dataset/data.yaml, degrees=0.0, deterministic=True, device=auto, dfl=1.662770506590975, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0007475524102215215, lrf=0.005463484813185952, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, mome

In [ ]:
# This script performs the final YOLOv8 training run using the best
# hyperparameters found during the previous optimization step.

from ultralytics import YOLO
import os
import yaml

def train_final_model(
    dataset_path: str,
    project_name: str,
    epochs: int,
    batch_size: int,
    image_size: int,
    patience: int,
    model_size: str,
    best_params: dict
):
    """
    Train a YOLOv8 segmentation model with a fixed set of hyperparameters.

    Args:
        dataset_path (str): Path to the YOLO dataset directory.
        project_name (str): Name for the training project.
        epochs (int): Number of training epochs.
        batch_size (int): Batch size for training.
        image_size (int): Image size for training.
        patience (int): Early stopping patience.
        model_size (str): The YOLO model variant (e.g., 'n', 's', 'm').
        best_params (dict): Dictionary of optimized hyperparameters.
    """
    print("🚀 Starting final training with best hyperparameters...")

    # Load the base model
    model_name = f'yolov8{model_size}-seg.pt'
    model = YOLO(model_name)
    print(f"📥 Loaded {model_name}")

    # The data.yaml file should be inside your dataset path
    data_yaml_path = os.path.join(dataset_path, 'data.yaml')
    if not os.path.exists(data_yaml_path):
        print(f"❌ Error: data.yaml not found at {data_yaml_path}")
        return

    # Use a different name for this final run to keep things organized
    run_name = 'final_run'

    try:
        results = model.train(
            data=data_yaml_path,
            epochs=epochs,
            batch=batch_size,
            imgsz=image_size,
            device='auto', # Let YOLOv8 auto-detect GPU/CPU
            patience=patience,
            project=project_name,
            name=run_name,
            exist_ok=True,
            plots=True,
            cache=True,
            single_cls=False,
            optimizer='auto',
            cos_lr=True,
            lr0=best_params['lr0'],
            lrf=best_params['lrf'],
            weight_decay=best_params['weight_decay'],
            warmup_epochs=best_params['warmup_epochs'],
            box=best_params['box'],
            cls=best_params['cls'],
            dfl=best_params['dfl']
        )
        print("✅ Final training completed successfully!")

        # Display the final validation results
        print("\n📈 Final Validation Results:")
        metrics = results.results_dict

        # --- ADDED ERROR HANDLING HERE ---
        try:
            mAP50_S = metrics.get('metrics/mAP50(S)', 'N/A')
            mAP50_95_S = metrics.get('metrics/mAP50-95(S)', 'N/A')
            print(f"🎯 mAP50(S): {mAP50_S:.4f}" if isinstance(mAP50_S, (int, float)) else f"🎯 mAP50(S): {mAP50_S}")
            print(f"🎯 mAP50-95(S): {mAP50_95_S:.4f}" if isinstance(mAP50_95_S, (int, float)) else f"🎯 mAP50-95(S): {mAP50_95_S}")
        except Exception as metric_e:
            print(f"❌ Could not display metrics due to formatting error: {metric_e}")
            print(f"Raw metrics: {metrics}")
        # --- END OF ADDED ERROR HANDLING ---

    except Exception as e:
        print(f"❌ Training failed: {e}")

def test_specific_images(
    model_path: str,
    images_base_path: str,
    image_filenames: list,
    output_dir: str = 'test_results'
):
    """
    Runs prediction on a specific set of images using a trained model.

    Args:
        model_path (str): Path to the trained model file (e.g., 'path/to/best.pt').
        images_base_path (str): The base directory containing the test images.
        image_filenames (list): A list of image filenames from the test set.
        output_dir (str): Directory to save the prediction results.
    """
    print(f"\n🔬 Starting prediction on {len(image_filenames)} specific images...")

    # Check if the model file exists
    if not os.path.exists(model_path):
        print(f"❌ Error: Model file not found at {model_path}")
        return

    # Load the trained model
    model = YOLO(model_path)
    print(f"📥 Loaded model from {model_path}")

    # Construct the full paths for the images to test
    image_paths = [os.path.join(images_base_path, filename) for filename in image_filenames]

    # Filter out any non-existent image paths
    existing_paths = [p for p in image_paths if os.path.exists(p)]
    if not existing_paths:
        print(f"❌ Error: No images found at the specified paths in {images_base_path}")
        return

    print(f"Found {len(existing_paths)} images to process.")

    try:
        results = model.predict(
            source=existing_paths,
            save=True, # Save the output with boxes and masks
            project=output_dir,
            name='yolo_test_run'
        )
        print("✅ Prediction completed successfully!")
    except Exception as e:
        print(f"❌ Prediction failed: {e}")

if __name__ == '__main__':
    # --- Configuration ---

    # Path to the YOLO dataset created by the conversion script
    YOLO_DATASET_PATH = '/content/drive/MyDrive/yolo_segmentation_dataset'

    # Project and run names for the training output
    PROJECT_NAME = 'coco_segmentation_training'

    # Settings from your previous training pipeline
    FINAL_EPOCHS = 200
    BATCH_SIZE = 16
    IMAGE_SIZE = 640
    MODEL_SIZE = 'n'
    PATIENCE = 30

    # Best hyperparameters from your previous log
    BEST_HYPERPARAMS = {
        'lr0': 0.0007475524102215215,
        'lrf': 0.005463484813185952,
        'weight_decay': 0.00014275735675228264,
        'warmup_epochs': 2,
        'box': 6.79718182417978,
        'cls': 0.237698606960847,
        'dfl': 1.662770506590975
    }

    # --- FINAL TRAINING RUN ---
    # This section is commented out to prevent re-training every time you run the script.
    # To re-train, simply uncomment the lines below.

    # train_final_model(
    #     dataset_path=YOLO_DATASET_PATH,
    #     project_name=PROJECT_NAME,
    #     epochs=FINAL_EPOCHS,
    #     batch_size=BATCH_SIZE,
    #     image_size=IMAGE_SIZE,
    #     patience=PATIENCE,
    #     model_size=MODEL_SIZE,
    #     best_params=BEST_HYPERPARAMS
    # )

    # --- SPECIFIC IMAGE TESTING ---
    # To run predictions on specific images,
    # 1. Ensure a trained model exists (e.g., 'best.pt').
    # 2. Add the filenames of the images you want to test to the list below.

    # CORRECTED PATH to find the trained model
    MODEL_PATH = os.path.join(PROJECT_NAME, 'final_run', 'weights', 'best.pt')

    # --- Path to the images you want to test
    # UPDATE THIS PATH to point to your test-30 folder
    TEST_IMAGES_BASE_PATH = '/content/drive/MyDrive/coco2017/test-30'

    TEST_IMAGES_TO_EVALUATE = [
        '000000001551.jpg',
    ]

    test_specific_images(
        model_path=MODEL_PATH,
        images_base_path=TEST_IMAGES_BASE_PATH,
        image_filenames=TEST_IMAGES_TO_EVALUATE,
    )



🔬 Starting prediction on 1 specific images...
📥 Loaded model from coco_segmentation_training/final_run/weights/best.pt
Found 1 images to process.

0: 448x640 3 persons, 84.1ms
Speed: 1.7ms preprocess, 84.1ms inference, 2.7ms postprocess per image at shape (1, 3, 448, 640)
Results saved to test_results/yolo_test_run3
✅ Prediction completed successfully!
